**Wan 2.1 1.3B Studio persistent worker.** One long-running session that owns the whole job pipeline: a feeder that polls the jobs dataset, a conditioning-blob stage, and TWO ComfyUI samplers — one per T4 — plus the decode/QA/upload tail.

## Model: Wan 2.1 1.3B (silent text-to-video, owner request 2026-10-21 — T44)

Replaced LTX-2.5 the same day it shipped: the owner picked Alibaba's small Wan. Same studio contract (jobs dataset in, `job_<id>.mp4` out, insurance pushes, Telegram mirror, cmd/merge terminal modes), new stack — every file PUBLIC in `Comfy-Org/Wan_2.1_ComfyUI_repackaged`, vanilla ComfyUI nodes, no GGUF, no custom loaders:

| piece | file | size |
|---|---|---|
| DiT (text-to-video) | `wan2.1_t2v_1.3B_fp16.safetensors` | 2.64 GiB |
| DiT (keyframe shots) | `wan2.1_fun_inp_1.3B_bf16.safetensors` | 2.91 GiB |
| text encoder | `umt5_xxl_fp8_e4m3fn_scaled.safetensors` | 6.27 GiB |
| VAE | `wan_2.1_vae.safetensors` | 0.24 GiB |

Honest contract changes vs LTX-2.5: the mp4 is **silent** (Wan 2.1 has no audio stream), **480P at 16 fps**, and durations ride the 81-frame native window (5 s) with 10 s on the 161-frame extrapolation. In exchange the downloads are 4x lighter, the encoder fits a T4, and both DiTs fit VRAM side by side — the retry ladder shrinks geometry instead of swapping quants.

Keyframe shots (manifest `frame_real: true`) run the Fun-InP variant through the vanilla `WanFunInpaintToVideo` node: the video starts exactly on the shipped keyframe. Shots without a real frame run pure text-to-video on the fp16 checkpoint.

## Why the conditioning blob survives the model swap

The three-stage split stays because it is proven, not because Wan needs it: the encode stage still runs ALONE on a dedicated server, encoding each shot's positive/negative conditioning (and, for Fun-InP, the encoded keyframe + mask baked into that conditioning) into small `.pt` blobs; each sampler then loads blobs and runs the official ComfyUI schedule in ONE graph: KSampler 30 steps uni_pc/simple cfg 6 over ModelSamplingSD3 shift 8, then tiled VAE decode and a silent MP4 through CreateVideo + SaveVideo — no separate decode process, no audio VAE.

The frame grid changes with the model: Wan wants `frames % 4 == 1` at 16 fps (LTX wanted `n % 8 == 1` at 25). The server ships 5 s -> 81 frames, 10 s -> 161 frames; gears 0.4/0.3/0.2 map to 832x480 / 704x400 / 480x272.


In [ ]:
# Stage 0 - configuration, environment, and the conditioning key scheme.
# T44: owner request (2026-10-21) - LTX-2.5 is out, the studio runs
# **Wan 2.1 1.3B** (Alibaba's Apache-2.0 text-to-video DiT). Same pipeline
# contract (jobs dataset in, job_<id>.mp4 out, insurance, Telegram mirror,
# cmd/merge terminal modes), new stack - every file PUBLIC in
# Comfy-Org/Wan_2.1_ComfyUI_repackaged, no gates, no GGUF, no custom loaders:
#   * DiT: wan2.1_t2v_1.3B_fp16 (2.64 GiB) - the official ComfyUI example uses
#     exactly this file and says fp16 > bf16 > fp8 on quality.
#   * Keyframe shots: wan2.1_fun_inp_1.3B_bf16 (2.91 GiB) - the Fun-InP variant
#     of the SAME 1.3B that consumes a start image through the vanilla
#     WanFunInpaintToVideo node, so the studio keeps its keyframe UX.
#   * Text encoder: umt5_xxl_fp8_e4m3fn_scaled (6.27 GiB) - 4x lighter than the
#     LTX gemma4 encoder, fits a T4 next to the samplers.
#   * VAE: wan_2.1_vae (0.24 GiB). WAN 2.1 IS SILENT - there is no audio VAE,
#     the mp4 has no audio track (honest regression vs LTX-2.5, owner-decided).
# Sampling (official ComfyUI text_to_video_wan.json): KSampler 30 steps,
# cfg 6.0, uni_pc / simple, ModelSamplingSD3 shift 8.0, EmptyHunyuanLatentVideo
# geometry 832x480, frame counts on the n % 4 == 1 grid at 16 fps.
import hashlib
import json
import os
import re
import shutil
import signal
import subprocess
import sys
import threading
import time
import uuid
from pathlib import Path

import psutil

# Compute dtype: the T4 (sm75) has no bf16 tensor cores; comfy stores fp16
# weights natively and casts the one bf16 file (fun_inp) to fp16 on load.
UNET_DTYPE_FLAGS = {}

COMFY_REPO = "https://github.com/Comfy-Org/ComfyUI.git"
COMFY_COMMIT = "b1a247c9cf9dbfda4b2c0a0c1d50950667eeaa0c"  # proven on Kaggle in T42/T43

# --- model sources (all PUBLIC, no HF token) ---------------------------------
MODEL_REPO = "Comfy-Org/Wan_2.1_ComfyUI_repackaged"
os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"

def _f(subdir, file, size, sha):
    return {"repo": MODEL_REPO, "file": f"split_files/{subdir}/{file}",
            "hf_filename": file, "bytes": size, "sha256": sha}

# sizes + sha256 are the HF LFS oids read from Comfy-Org/Wan_2.1_ComfyUI_repackaged (2026-10-21).
DIT_T2V = _f("diffusion_models", "wan2.1_t2v_1.3B_fp16.safetensors",
             2838303560, "be531024cd9018cb5b48c40cfbb6a6191645b1c792eb8bf4f8c1c6e10f924dc5")
DIT_FUN_INP = _f("diffusion_models", "wan2.1_fun_inp_1.3B_bf16.safetensors",
                 3128957992, "8495d2b1673ffb18abb548a64ff3b0e4bd367734f653096f7a8a3ad46954d511")
TEXT_ENCODER = _f("text_encoders", "umt5_xxl_fp8_e4m3fn_scaled.safetensors",
                  6735906897, "c3355d30191f1f066b26d93fba017ae9809dce6c627dda5f6a66eaa651204f68")
VIDEO_VAE = _f("vae", "wan_2.1_vae.safetensors",
               253815318, "2fc39d31359a4b0a64f55876d8ff7fa8d780956ae2cb13463b0223e15148976b")
# All four files download upfront: the whole stack is 9.4 GiB (LTX was 40+),
# so the retry ladder needs no lazy quant - both DiTs fit one T4 side by side.

# --- sampling configuration (official ComfyUI Wan T2V example) ----------------
FPS = 16
NEGATIVE_PROMPT = ("色调艳丽，过曝，静态，细节模糊不清，字幕，风格，作品，画作，画面，静止，整体发灰，"
                   "最差质量，低质量，JPEG压缩残留，丑陋的，残缺的，多余的手指，画得不好的手部，"
                   "画得不好的脸部，畸形的，毁容的，形态畸形的肢体，手指融合，静止不动的画面，"
                   "杂乱的背景，三条腿，背景人很多，倒着走")
STEPS = 30
CFG = 6.0
SAMPLER_NAME = "uni_pc"
SCHEDULER = "simple"
SHIFT = 8.0          # official 1.3B guidance: sample_shift 8 (range 8-12)

# The resolution gears (both dimensions divisible by 16, both ~16:9 to match
# the 832x480 keyframe canvas the server ships). Wan 1.3B is trained at
# 832x480 480P; other sizes work but are off-distribution, so the fallback
# gear SHRINKS rather than scales up.
GEAR_TABLE = {
    "0.2": (480, 272),
    "0.3": (704, 400),
    "0.4": (832, 480),
}
GEAR = "0.4"  # @param ["0.2", "0.3", "0.4"]
if "GEAR_OVERRIDE" in globals():
    GEAR = str(globals()["GEAR_OVERRIDE"])
    print(f"GEAR_OVERRIDE active: GEAR = {GEAR} (campaign run)")
CLIP_SECONDS = 5.0  # @param {type:"number"}


def frame_count(seconds):
    """Frames for a duration on the Wan grid at FPS.

    The latent node takes length with step 4 and the sampler wants the
    n % 4 == 1 shape the VAE was trained on (81 f = 5.06 s native window;
    10 s shots ride the 161-frame extrapolation, quality is softer there).
    """
    n = max(9, round(seconds * FPS))
    return n + ((1 - (n % 4)) % 4)


if GEAR not in GEAR_TABLE:
    raise RuntimeError(f"Unknown GEAR {GEAR!r}; pick one of {sorted(GEAR_TABLE)}")
GEOMETRY = {"width": GEAR_TABLE[GEAR][0], "height": GEAR_TABLE[GEAR][1],
            "length": frame_count(CLIP_SECONDS)}

CONTENT = Path("/kaggle/tmp") if Path("/kaggle").is_dir() else Path("/content")
WORK = CONTENT / "wan13_sample_probe"
COMFY_DIR = CONTENT / "ComfyUI"
GGUF_NODE_DIR = COMFY_DIR / "custom_nodes" / "ComfyUI-GGUF"  # NOT installed for Wan
CONTROL_NODE_DIR = COMFY_DIR / "custom_nodes" / "H3BenchmarkControl"  # kept name: shared with the test worker
MODEL_STORE = WORK / "models"
CONDITIONING_STORE = WORK / "conditioning"
LATENT_STORE = WORK / "latents"
OUTPUT_DIR = COMFY_DIR / "output"
COMFY_LOG = WORK / "comfyui.log"
RESULT_PATH = WORK / "sample_probe_results.json"
# /kaggle/working replaces Drive as the persistent store (blob handoff).
DRIVE_CONDITIONING_STORE = Path("/kaggle/working") / "conditioning" \
    if Path("/kaggle/working").is_dir() else WORK / "conditioning_persist"
DRIVE_VIDEO_STORE = Path("/kaggle/working") / "out" \
    if Path("/kaggle/working").is_dir() else WORK / "out"
for directory in (WORK, MODEL_STORE, CONDITIONING_STORE, LATENT_STORE):
    directory.mkdir(parents=True, exist_ok=True)

GIB = 1024 ** 3
BASE_URL = "http://127.0.0.1:8188"
RUN_ID = time.strftime("%Y%m%dT%H%M%SZ", time.gmtime())

if RESULT_PATH.exists():
    backup = RESULT_PATH.with_suffix(f".json.bak-{time.strftime('%Y%m%dT%H%M%S')}")
    shutil.copy2(RESULT_PATH, backup)
    try:
        PROBE = json.loads(RESULT_PATH.read_text(encoding="utf-8"))
        print(f"Resumed earlier results ({sorted(PROBE.get('stages', {}))}); backup {backup.name}")
    except json.JSONDecodeError:
        PROBE = {"schema_version": 3, "stages": {}}
else:
    PROBE = {"schema_version": 3, "stages": {}}
PROBE.setdefault("stages", {})


def save():
    RESULT_PATH.write_text(json.dumps(PROBE, indent=2, ensure_ascii=False, default=str),
                           encoding="utf-8")


def utc_now():
    import datetime as dt
    return dt.datetime.now(dt.timezone.utc).isoformat()


CHILD_ENV_ALLOWLIST = {
    "PATH", "HOME", "LANG", "TMPDIR", "CUDA_HOME", "CUDA_PATH", "CUDA_VISIBLE_DEVICES",
    "CUDA_MODULE_LOADING", "CUDA_LAUNCH_BLOCKING", "LD_LIBRARY_PATH", "PYTHONPATH",
    "PYTHONUNBUFFERED", "PYTORCH_CUDA_ALLOC_CONF", "TRITON_CACHE_DIR", "XDG_CACHE_HOME",
    "SSL_CERT_FILE", "REQUESTS_CA_BUNDLE", "PIP_CACHE_DIR",
}
SECRET_NAME_RE = re.compile(r"TOKEN|SECRET|PASSWORD|API_KEY|AUTHORIZATION|CREDENTIAL", re.I)


def scrubbed_child_environment(extra=None):
    merged = {name: value for name, value in os.environ.items()
              if name in CHILD_ENV_ALLOWLIST and not SECRET_NAME_RE.search(name)}
    merged.setdefault("PATH", "/usr/local/bin:/usr/bin:/bin")
    merged.setdefault("HOME", "/root")
    merged["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"
    merged["PIP_DISABLE_PIP_VERSION_CHECK"] = "1"
    if extra:
        rejected = [n for n in extra if n not in CHILD_ENV_ALLOWLIST or SECRET_NAME_RE.search(n)]
        if rejected:
            raise RuntimeError(f"Refusing secret-like child environment entries: {rejected}")
        merged.update(extra)
    return merged


def run(command, cwd=None, env=None):
    print("$ " + " ".join(str(part) for part in command))
    return subprocess.run(command, cwd=cwd, env=scrubbed_child_environment(env), check=True)


def sha256_file(path, chunk_bytes=16 * 1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(chunk_bytes), b""):
            digest.update(block)
    return digest.hexdigest()


import torch  # noqa: E402

frames = GEOMETRY["length"]
if (frames - 1) % 4 != 0:
    raise RuntimeError(f"Wan frame count must satisfy n % 4 == 1; got {frames}")
if GEOMETRY["width"] % 16 or GEOMETRY["height"] % 16:
    raise RuntimeError(f"Wan dimensions must be divisible by 16: {GEOMETRY}")
LATENT_T = ((frames - 1) // 4) + 1
EXPECTED_LATENT_SHAPES = [
    [1, 16, LATENT_T, GEOMETRY["height"] // 8, GEOMETRY["width"] // 8],
]


# The blob's identity is its inputs. Recomputing the key here means a name
# match is itself proof that repo, variant, encoder, prompts, geometry and
# fps agree. variant = "t2v" | "fun_inp" - the Fun-InP conditioning carries
# the encoded keyframe, so its blob must never satisfy a t2v request.
def conditioning_key_for(prompt, frame_sha, geometry, variant):
    payload = {
        "model": f"Wan2.1-1.3B-{variant}",
        "te_sha256": TEXT_ENCODER["sha256"][:32],
        "pos_sha256": hashlib.sha256(prompt.encode("utf-8")).hexdigest(),
        "neg_sha256": hashlib.sha256(NEGATIVE_PROMPT.encode("utf-8")).hexdigest(),
        "fps": FPS, "shift": SHIFT,
        **geometry,
    }
    if frame_sha is not None:
        payload["frame_sha256"] = frame_sha
    return "cond-" + hashlib.sha256(json.dumps(
        payload, sort_keys=True, separators=(",", ":")).encode("utf-8")).hexdigest()[:32]


def negative_key(variant):
    return "neg-" + hashlib.sha256(json.dumps({
        "model": f"Wan2.1-1.3B-{variant}",
        "te_sha256": TEXT_ENCODER["sha256"][:32],
        "neg_sha256": hashlib.sha256(NEGATIVE_PROMPT.encode("utf-8")).hexdigest(),
        "fps": FPS,
    }, sort_keys=True, separators=(",", ":")).encode("utf-8")).hexdigest()[:32]


virtual_memory = psutil.virtual_memory()
disk = shutil.disk_usage(str(CONTENT))
PROBE["stages"]["environment"] = {
    "measured_at": utc_now(),
    "run_id": RUN_ID,
    "host_ram_total_gib": round(virtual_memory.total / GIB, 3),
    "host_ram_available_gib": round(virtual_memory.available / GIB, 3),
    "cpu_count": os.cpu_count(),
    "disk_free_gib": round(disk.free / GIB, 3),
    "python": sys.version.split()[0],
    "torch": str(torch.__version__),
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "compute_capability": ".".join(str(x) for x in torch.cuda.get_device_capability(0))
    if torch.cuda.is_available() else None,
    "vram_total_mib": round(torch.cuda.get_device_properties(0).total_memory / 1024 ** 2, 1)
    if torch.cuda.is_available() else None,
    "model": "Wan 2.1 1.3B (silent, no audio track)",
    "dit_t2v_gib": round(DIT_T2V["bytes"] / GIB, 3),
    "dit_fun_inp_gib": round(DIT_FUN_INP["bytes"] / GIB, 3),
    "text_encoder_gib": round(TEXT_ENCODER["bytes"] / GIB, 3),
    "vae_gib": round(VIDEO_VAE["bytes"] / GIB, 3),
    "steps": STEPS, "cfg": CFG, "sampler": SAMPLER_NAME, "scheduler": SCHEDULER,
    "shift": SHIFT,
    "fps": FPS, "negative": NEGATIVE_PROMPT[:40] + "...",
    "gear": GEAR, "geometry": GEOMETRY, "clip_seconds": CLIP_SECONDS,
    "expected_latent_shapes": EXPECTED_LATENT_SHAPES,
}
save()

print(json.dumps(PROBE["stages"]["environment"], indent=2, ensure_ascii=False))
print()
print("=" * 72)
print(f"available RAM      : {virtual_memory.available / GIB:.2f} GiB")
print(f"DiT t2v            : {DIT_T2V['bytes'] / GIB:.2f} GiB fp16 "
      f"(fun_inp keyframe variant {DIT_FUN_INP['bytes'] / GIB:.2f} GiB)")
print(f"text encoder       : {TEXT_ENCODER['bytes'] / GIB:.2f} GiB fp8-scaled")
print(f"VAE                : {VIDEO_VAE['bytes'] / GIB:.2f} GiB")
print(f"schedule           : {STEPS} steps, {SAMPLER_NAME}/{SCHEDULER}, "
      f"cfg {CFG}, shift {SHIFT}")
print(f"geometry / fps     : {GEOMETRY['width']}x{GEOMETRY['height']} / "
      f"{frames} frames @ {FPS} ({CLIP_SECONDS} s)")
print(f"disk free          : {disk.free / GIB:.1f} GiB")
print("=" * 72)

# --- [studio-telemetry] live phase markers -> outputs dataset (progress.json) --
import threading as _tth
_TEL_LOCK = _tth.Lock()
_TEL_LAST = {"t": 0.0}
_TPROG_T0 = time.time()
PROG_EVENTS = []
def _tel_body(tag):
    try:
        out_dir = Path("/kaggle/working/out")
        out_dir.mkdir(parents=True, exist_ok=True)
        (out_dir / "progress.json").write_text(json.dumps(
            {"events": PROG_EVENTS[-80:]}, ensure_ascii=False, indent=1), encoding="utf-8")
        meta = out_dir / "dataset-metadata.json"
        if not meta.exists():
            json.dump({"title": "rmc-studio-outputs", "id": "freedomform/rmc-studio-outputs",
                       "licenses": [{"name": "CC0-1.0"}]}, open(meta, "w"))
        os.environ["KAGGLE_API_TOKEN"] = "KGAT_abba3bba83ed8f6a11855f7a819228ea"
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kaggle"],
                       capture_output=True, timeout=300)
        created = subprocess.run(
            [sys.executable, "-m", "kaggle", "datasets", "create", "-p", str(out_dir),
             "--dir-mode", "zip"], capture_output=True, text=True, timeout=1200)
        if "successfully created" not in (created.stdout or "").lower():
            version = subprocess.run(
                [sys.executable, "-m", "kaggle", "datasets", "version",
                 "-p", str(out_dir), "--dir-mode", "zip",
                 "-m", f"prog {tag} {time.strftime('%H:%M')}"],
                capture_output=True, text=True, timeout=1200)
            print(f"[tel] push rc={version.returncode}", flush=True)
    except Exception as exc:
        print(f"[tel] push failed: {type(exc).__name__}: {exc}", flush=True)
def prog(tag):
    ev = {"s": round(time.time() - _TPROG_T0, 1), "tag": tag}
    PROG_EVENTS.append(ev)
    print(f"[tel] +{ev['s']}s {tag}", flush=True)
    with _TEL_LOCK:
        if time.time() - _TEL_LAST["t"] < 120:
            return
        _TEL_LAST["t"] = time.time()
    def _go():
        lk = globals().get("PUSH_LOCK")
        if lk is not None:
            with lk:
                _tel_body(tag)
        else:
            _tel_body(tag)
    _tth.Thread(target=_go, daemon=True).start()
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "kaggle"],
               capture_output=True, timeout=600)
prog("cell1 config done (Wan 2.1 1.3B)")


In [ ]:
# Stage 1 - ComfyUI at the pinned commit, the studio control nodes,
prog("cell2 comfyui setup begin")
# and an EARLY node-availability check (CPU server, no models): every node the
# Wan graphs use must exist in the pinned ComfyUI before anything downloads.
# No ComfyUI-GGUF here: Wan 2.1 1.3B runs on the stock fp16 safetensors
# loader; the only custom nodes are the slim studio control nodes.
import importlib.metadata as importlib_metadata
import tempfile

setup_started = time.perf_counter()
PROTECTED = {"torch", "torchvision", "torchaudio", "triton"}


def canonical(name):
    return re.sub(r"[-_.]+", "-", str(name)).lower()


def is_protected(name):
    return canonical(name) in PROTECTED or canonical(name).startswith("nvidia-")


def protected_snapshot():
    snapshot = {}
    for distribution in importlib_metadata.distributions():
        raw = distribution.metadata.get("Name")
        if raw and is_protected(raw):
            snapshot[canonical(raw)] = str(distribution.version)
    return dict(sorted(snapshot.items()))


def ensure_checkout(repo, destination, commit, allow_dirty=False):
    destination = Path(destination)
    if not destination.exists():
        run(["git", "clone", "--filter=blob:none", repo, str(destination)])
    if not allow_dirty:
        dirty = subprocess.check_output(
            ["git", "-C", str(destination), "status", "--porcelain", "--untracked-files=no"],
            text=True, env=scrubbed_child_environment()).strip()
        if dirty:
            raise RuntimeError(f"Refusing a dirty checkout: {destination}: {dirty[:1000]}")
    head = subprocess.check_output(["git", "-C", str(destination), "rev-parse", "HEAD"],
                                   text=True, env=scrubbed_child_environment()).strip()
    if head != commit:
        run(["git", "-C", str(destination), "fetch", "origin", commit])
        run(["git", "-C", str(destination), "checkout", "--force", "--detach", commit])
    head = subprocess.check_output(["git", "-C", str(destination), "rev-parse", "HEAD"],
                                   text=True, env=scrubbed_child_environment()).strip()
    if head != commit:
        raise RuntimeError(f"Checkout verification failed: HEAD={head}")


def pip_install_without_touching_torch(requirements_path, label):
    """Resolve first, refuse the plan if it would move torch, then install exactly."""
    before = protected_snapshot()
    handle, name = tempfile.mkstemp(prefix=f"wan-{label}-", suffix=".json")
    os.close(handle)
    report_path = Path(name)
    try:
        run([sys.executable, "-m", "pip", "install", "-q", "--dry-run",
             "--report", str(report_path), "-r", str(requirements_path)])
        plan = json.loads(report_path.read_text(encoding="utf-8"))
    finally:
        report_path.unlink(missing_ok=True)
    actions = []
    for item in plan.get("install", []):
        metadata = item.get("metadata", {})
        pkg, version = canonical(metadata.get("name", "")), str(metadata.get("version", ""))
        if not re.fullmatch(r"[a-z0-9][a-z0-9-]*", pkg) or not version:
            raise RuntimeError(f"Unsafe package in {label} plan: {pkg!r} {version!r}")
        actions.append({"name": pkg, "spec": f"{pkg}=={version}"})
    blocked = [a for a in actions if is_protected(a["name"])]
    if blocked:
        raise RuntimeError(
            f"Refusing the {label} plan: it would mutate protected runtime distributions "
            f"{[a['spec'] for a in blocked]}. Every measurement in this project assumes "
            "the stock torch/triton stack.")
    if actions:
        run([sys.executable, "-m", "pip", "install", "-q", "--no-deps",
             *[a["spec"] for a in actions]])
    after = protected_snapshot()
    if before != after:
        raise RuntimeError(f"Protected distributions changed during {label}: {before} -> {after}")
    return {"installed": [a["spec"] for a in actions], "protected_unchanged": True}


ensure_checkout(COMFY_REPO, COMFY_DIR, COMFY_COMMIT)
comfy_install = pip_install_without_touching_torch(COMFY_DIR / "requirements.txt", "comfyui")
print(f"installed comfyui:{len(comfy_install['installed'])} packages, torch untouched")

# --- Wan node sanity check at the pinned commit -------------------------------
# The graphs rely on WanTextEncoder loading through CLIPLoader(type=wan),
# WanFunInpaintToVideo for keyframe shots, and ModelSamplingSD3 for the flow
# shift - all vanilla. Assert the three names exist in the source tree so a
# ComfyUI refactor cannot silently break the session.
for node_file, marker in (("comfy_extras/nodes_wan.py", "WanFunInpaintToVideo"),
                          ("comfy_extras/nodes_model_advanced.py", "ModelSamplingSD3"),
                          ("comfy_extras/nodes_hunyuan.py", "EmptyHunyuanLatentVideo")):
    text = (COMFY_DIR / node_file).read_text(encoding="utf-8")
    if marker not in text:
        raise RuntimeError(f"{node_file} at the pinned commit has no {marker}; "
                           "the Wan graphs would not build. Inspect the checkout.")

custom_nodes = COMFY_DIR / "custom_nodes"
allowed = {"H3BenchmarkControl", "__pycache__",
           "example_node.py.example", "websocket_image_save.py"}
unexpected = [p.name for p in custom_nodes.iterdir() if p.name not in allowed]
if unexpected:
    raise RuntimeError(f"Unexpected custom nodes in a controlled runtime: {unexpected}")

CONTROL_NODE_DIR.mkdir(parents=True, exist_ok=True)
# Slim studio control nodes: blob + latent save/load (plain torch.save of the
# Wan conditioning/latent dicts - plain tensors, no nested-tensor restore
# dance) and a finite-check latent probe.
(CONTROL_NODE_DIR / "__init__.py").write_text(r'''
import json
import logging
from pathlib import Path

import torch


def _detach_to_cpu(obj):
    if isinstance(obj, torch.Tensor):
        return obj.detach().cpu()
    if isinstance(obj, dict):
        return {key: _detach_to_cpu(value) for key, value in obj.items()}
    if isinstance(obj, (list, tuple)):
        return type(obj)(_detach_to_cpu(value) for value in obj)
    return obj


class H3SaveConditioning:
    """Persist {conditioning, samples} so later runs never load the encoder.

    For Wan the CONDITIONING is the CLIPTextEncode output (a list of
    [embeddings, options] pairs; Fun-InP options carry the encoded keyframe)
    and samples is the zeros video latent - both plain python/torch containers.
    """

    @classmethod
    def INPUT_TYPES(cls):
        return {"required": {
            "conditioning": ("CONDITIONING",), "samples": ("LATENT",),
            "store_dir": ("STRING", {"default": ""}),
            "key": ("STRING", {"default": "conditioning"})}}
    RETURN_TYPES = ()
    FUNCTION = "save"
    CATEGORY = "H3 benchmark/control"
    OUTPUT_NODE = True

    def save(self, conditioning, samples, store_dir, key):
        directory = Path(store_dir)
        directory.mkdir(parents=True, exist_ok=True)
        payload = {"schema_version": 2,
                   "conditioning": _detach_to_cpu(conditioning),
                   "samples": _detach_to_cpu(samples)}
        destination = directory / f"{key}.pt"
        staged = destination.with_suffix(".pt.tmp")
        torch.save(payload, staged)
        staged.replace(destination)
        logging.info("WAN_SAVE_CONDITIONING %s", json.dumps(
            {"key": key, "bytes": destination.stat().st_size}, separators=(",", ":")))
        return {}


class H3LoadConditioning:
    """Return a precomputed (CONDITIONING, LATENT) pair without a CLIPLoader."""

    @classmethod
    def INPUT_TYPES(cls):
        return {"required": {
            "store_dir": ("STRING", {"default": ""}),
            "key": ("STRING", {"default": "conditioning"}),
        }}
    RETURN_TYPES = ("CONDITIONING", "LATENT")
    RETURN_NAMES = ("conditioning", "samples")
    FUNCTION = "load"
    CATEGORY = "H3 benchmark/control"

    def load(self, store_dir, key):
        source = Path(store_dir) / f"{key}.pt"
        if not source.is_file():
            raise RuntimeError(f"Precomputed conditioning is missing: {source}")
        payload = torch.load(source, map_location="cpu", weights_only=False)
        if payload.get("schema_version") != 2:
            raise RuntimeError(f"Unsupported schema: {payload.get('schema_version')}")
        logging.info("WAN_LOAD_CONDITIONING %s", json.dumps(
            {"key": key, "bytes": source.stat().st_size}, separators=(",", ":")))
        return (payload["conditioning"], payload["samples"])


class H3SaveLatent:
    """Persist a LATENT dict (the sampled video latent) between stages."""

    @classmethod
    def INPUT_TYPES(cls):
        return {"required": {
            "samples": ("LATENT",),
            "store_dir": ("STRING", {"default": ""}),
            "key": ("STRING", {"default": "latent"})}}
    RETURN_TYPES = ("LATENT",)
    RETURN_NAMES = ("samples",)
    FUNCTION = "save"
    CATEGORY = "H3 benchmark/control"
    OUTPUT_NODE = True

    def save(self, samples, store_dir, key):
        directory = Path(store_dir)
        directory.mkdir(parents=True, exist_ok=True)
        payload = _detach_to_cpu(dict(samples))
        destination = directory / f"{key}.pt"
        staged = destination.with_suffix(".pt.tmp")
        torch.save(payload, staged)
        staged.replace(destination)
        logging.info("WAN_SAVE_LATENT %s", json.dumps(
            {"key": key, "bytes": destination.stat().st_size}, separators=(",", ":")))
        return (samples,)


class H3LoadLatent:
    """Load a LATENT dict saved by H3SaveLatent."""

    @classmethod
    def INPUT_TYPES(cls):
        return {"required": {
            "store_dir": ("STRING", {"default": ""}),
            "key": ("STRING", {"default": "latent"}),
        }}
    RETURN_TYPES = ("LATENT",)
    FUNCTION = "load"
    CATEGORY = "H3 benchmark/control"

    def load(self, store_dir, key):
        source = Path(store_dir) / f"{key}.pt"
        if not source.is_file():
            raise RuntimeError(f"Persisted latent is missing: {source}")
        payload = torch.load(source, map_location="cpu", weights_only=False)
        if not isinstance(payload, dict) or "samples" not in payload:
            raise RuntimeError(f"Persisted latent is malformed: {source}")
        return (payload,)


class H3LatentProbe:
    """Record the sampled latent and refuse non-finite values.

    A finite latent can still decode to black - the decode QA covers that -
    but a NaN here means the whole session is misconfigured, so fail fast.
    """

    @classmethod
    def INPUT_TYPES(cls):
        return {"required": {
            "samples": ("LATENT",),
            "filename_prefix": ("STRING", {"default": "wan13/probe"}),
        }}
    RETURN_TYPES = ("LATENT",)
    RETURN_NAMES = ("samples",)
    FUNCTION = "save_summary"
    CATEGORY = "H3 benchmark/control"
    OUTPUT_NODE = True

    def save_summary(self, samples, filename_prefix):
        import folder_paths
        packed = samples.get("samples")
        entries = []
        if isinstance(packed, torch.Tensor):
            entries.append(packed)
        elif isinstance(packed, (list, tuple)):
            entries += [p for p in packed if isinstance(p, torch.Tensor)]
        finite = [bool(torch.isfinite(p).all().item()) for p in entries]
        output_path = Path(folder_paths.get_output_directory()) / f"{filename_prefix}.latent_summary.json"
        output_path.parent.mkdir(parents=True, exist_ok=True)
        payload = {"shapes": [list(p.shape) for p in entries],
                   "dtypes": [str(p.dtype) for p in entries],
                   "finite": finite,
                   "keys": sorted(samples.keys())}
        output_path.write_text(json.dumps(payload, indent=2), encoding="utf-8")
        logging.info("WAN_LATENT_PROBE %s", json.dumps(payload["shapes"], separators=(",", ":")))
        if finite and not all(finite):
            raise RuntimeError(f"Sampled latent contains non-finite values: {payload}")
        return (samples,)


NODE_CLASS_MAPPINGS = {
    "H3SaveConditioning": H3SaveConditioning,
    "H3LoadConditioning": H3LoadConditioning,
    "H3SaveLatent": H3SaveLatent,
    "H3LoadLatent": H3LoadLatent,
    "H3LatentProbe": H3LatentProbe,
}
''')
print("studio control nodes written (slim Wan variant)")

FORBIDDEN_FLAGS = ("--use-sage-attention", "--highvram")


def supported_cli_flags():
    path = COMFY_DIR / "comfy" / "cli_args.py"
    text = path.read_text(encoding="utf-8") if path.is_file() else ""
    return set(re.findall(r'"(--[a-z0-9][a-z0-9-]*)"', text)) | \
        set(re.findall(r"'(--[a-z0-9][a-z0-9-]*)'", text))


# --- single-port server control (used by the verify pass and the blob stage;
# the two persistent SAMPLER servers in cell 4 have their own spawn_comfy) ----
COMFY_PROCESS = None
COMFY_LOG_HANDLE = None


def stop_comfy(term_timeout_s=15):
    global COMFY_PROCESS, COMFY_LOG_HANDLE
    if COMFY_PROCESS is not None and COMFY_PROCESS.poll() is None:
        os.killpg(COMFY_PROCESS.pid, signal.SIGTERM)
        try:
            COMFY_PROCESS.wait(timeout=term_timeout_s)
        except subprocess.TimeoutExpired:
            try:
                os.killpg(COMFY_PROCESS.pid, signal.SIGKILL)
                COMFY_PROCESS.wait(timeout=15)
            except (ProcessLookupError, subprocess.TimeoutExpired):
                pass
    COMFY_PROCESS = None
    if COMFY_LOG_HANDLE is not None:
        COMFY_LOG_HANDLE.close()
        COMFY_LOG_HANDLE = None


def start_comfy(timeout_s=900, quiet=False, port=8188, cpu=False, extra_flags=()):
    """Start one ComfyUI server. cpu=True forces CPU mode (node verification
    must not touch the GPUs; the blob stage leaves them to the samplers)."""
    global COMFY_PROCESS, COMFY_LOG_HANDLE
    stop_comfy()
    log_path = WORK / f"comfy_p{port}.log"
    COMFY_LOG_HANDLE = log_path.open("a", encoding="utf-8", buffering=1)
    supported = supported_cli_flags()
    command = [sys.executable, "main.py", "--listen", "127.0.0.1", "--port", str(port),
               "--disable-auto-launch", "--preview-method", "none", "--cache-classic"]
    for flag in list(extra_flags) + (["--cpu"] if cpu and "--cpu" in supported else []):
        if flag in supported:
            command.append(flag)
    for forbidden in FORBIDDEN_FLAGS:
        if forbidden in command:
            raise RuntimeError(f"{forbidden} is forbidden here.")
    print("$ " + " ".join(command))
    env = {"CUDA_VISIBLE_DEVICES": ""} if cpu else {}
    COMFY_PROCESS = subprocess.Popen(
        command, cwd=COMFY_DIR, env=scrubbed_child_environment({"PYTHONUNBUFFERED": "1", **env}),
        stdout=COMFY_LOG_HANDLE, stderr=subprocess.STDOUT, start_new_session=True)
    import requests
    deadline = time.time() + timeout_s
    while time.time() < deadline:
        if COMFY_PROCESS.poll() is not None:
            raise RuntimeError("ComfyUI exited during startup:\n"
                               + log_path.read_text(errors="replace")[-8000:])
        try:
            if requests.get(f"http://127.0.0.1:{port}/system_stats", timeout=5).status_code == 200:
                return f"http://127.0.0.1:{port}"
        except Exception:
            pass
        time.sleep(2)
    stop_comfy()
    raise RuntimeError(f"ComfyUI startup timeout on port {port}")


# --- EARLY node verification (CPU, no models) ---------------------------------
REQUIRED_NODES = {
    # loaders + text
    "CLIPLoader", "CLIPTextEncode", "VAELoader", "LoadImage", "UNETLoader",
    # Wan graph
    "EmptyHunyuanLatentVideo", "WanFunInpaintToVideo", "ModelSamplingSD3",
    "KSampler", "VAEDecode", "CreateVideo", "SaveVideo",
    # studio control
    "H3SaveConditioning", "H3LoadConditioning", "H3SaveLatent", "H3LoadLatent",
    "H3LatentProbe",
}

import requests  # noqa: E402
verify_base = start_comfy(port=8199, cpu=True, timeout_s=1200)
try:
    info = requests.get(f"{verify_base}/object_info", timeout=600).json()
    missing = sorted(REQUIRED_NODES - set(info))
    if missing:
        raise RuntimeError(f"Pinned ComfyUI is missing required nodes: {missing}")
    # the exact contract of the two graph-shaping inputs we depend on
    clip_types = info["CLIPLoader"]["input"]["required"]["type"][0]
    if "wan" not in clip_types:
        raise RuntimeError(f"CLIPLoader types lack 'wan': {clip_types}")
    inp_types = info["WanFunInpaintToVideo"]["input"]["required"]
    for name in ("positive", "negative", "vae", "width", "height", "length", "batch_size"):
        if name not in inp_types and name not in info["WanFunInpaintToVideo"]["input"].get("optional", {}):
            raise RuntimeError(f"WanFunInpaintToVideo lost its '{name}' input")
    PROBE["stages"]["node_check"] = {"verified": len(REQUIRED_NODES), "missing": []}
    print(f"node check: all {len(REQUIRED_NODES)} required nodes present "
          "(CLIPLoader type=wan, WanFunInpaintToVideo contract OK)")
finally:
    stop_comfy()
    print("verification server stopped")
save()
prog("cell2 done (comfy+nodes verified)")


In [ ]:
# Stage 2 (batch) -- model downloads + conditioning blobs for every shot.
prog("cell3 models begin")
#
# Downloads: both DiTs + the VAE always (9.4 GiB total - the whole stack fits
# where the LTX encoder alone did not). The 6.27 GiB text encoder downloads
# ONLY when at least one conditioning blob is missing.
#
# Blobs: for every shot the encode server (port 8190, alone on GPU 0) runs the
# Wan conditioning graph ONCE per geometry - positive + negative conditioning,
# and for keyframe shots the Fun-InP latent (zeros + encoded start image baked
# into the conditioning) - and saves both through H3SaveConditioning. The
# negative prompt is static, so its blob is shared across all shots of a
# variant (one key per session and variant).
import requests

def load_shot_manifest():
    candidates = [Path("/kaggle/input/rmc-studio-jobs") / "shots.json",
                  Path("/kaggle/input/rmc-ltx-t4x2-inputs") / "shots.json",
                  Path("/kaggle/input/rmc-h3-t4x2-inputs") / "shots.json",
                  WORK / "inputs" / "shots.json"]
    try:
        candidates += sorted(Path("/kaggle/input").glob("**/shots.json"))
    except Exception:
        pass
    for candidate in candidates:
        if candidate.is_file():
            shots = json.loads(candidate.read_text(encoding="utf-8"))
            print(f"shot manifest: {candidate} ({len(shots)} shots)")
            return shots, candidate.parent
    listed = sorted(str(p) for p in Path("/kaggle/input").glob("*")) \
        if Path("/kaggle/input").is_dir() else []
    raise RuntimeError("shots.json not found; attach the jobs dataset "
                       f"(/kaggle/input contains: {listed})")


def geometry_for(gear, frames):
    width, height = GEAR_TABLE[gear]
    return {"width": width, "height": height, "length": frames}


def variant_for(entry):
    """fun_inp when the server shipped a REAL user keyframe (frame_real) - the
    video then starts exactly on it. Neutral canvases and manifests from
    servers older than T44 fall back to pure text-to-video."""
    return "fun_inp" if entry.get("frame_real") else "t2v"


def dit_for(variant):
    return DIT_FUN_INP if variant == "fun_inp" else DIT_T2V


# [studio-worker] the feeder owns ALL job intake (single dedup point via
# done_ids.json + SEEN_JOBS); this cell only prepares the FIRST batch.
MANIFEST, SHOT_BASE = [], (WORK / "inputs")
SHOTS = []
for entry in MANIFEST:
    if entry.get("mode") in ("cmd", "merge"):
        continue  # handled by the feeder's handle_special path
    frame_path = SHOT_BASE / entry["frame"]
    if not frame_path.is_file():
        raise RuntimeError(f"keyframe missing: {frame_path}")
    variant = variant_for(entry)
    gears = sorted({entry.get("gear_primary", GEAR), entry.get("gear_fallback", "0.2")})
    geometries = {gear: geometry_for(gear, int(entry["frames"])) for gear in gears}
    frame_sha = hashlib.sha256(frame_path.read_bytes()).hexdigest() if variant == "fun_inp" else None
    SHOTS.append({
        "num": int(entry["num"]), "seconds": entry["seconds"],
        "frames": int(entry["frames"]), "prompt": entry["prompt"],
        "frame_path": frame_path, "has_frame": variant == "fun_inp",
        "variant": variant, "dit": dit_for(variant),
        "gear_primary": entry.get("gear_primary", GEAR),
        "gear_fallback": entry.get("gear_fallback", "0.2"),
        "geometries": geometries,
        "keys": {gear: conditioning_key_for(entry["prompt"], frame_sha, geometry, variant)
                 for gear, geometry in geometries.items()},
        "seed": 42 + int(entry["num"]) * 7919,
        "blobs": {},
    })
print("prepared shots: " + ", ".join(
    f"{s['num']:02d}/{s['frames']}f/{s['variant']}" for s in SHOTS))

NEG_KEYS = {variant: negative_key(variant) for variant in
            sorted({shot["variant"] for shot in SHOTS} | {"t2v", "fun_inp"})}


def find_blob_anywhere(key):
    try:
        return next(iter(Path("/kaggle/input").glob(f"**/{key}.pt")), None)
    except Exception:
        return None


def resolve_blob(key):
    resolved = next((p for p in (
        CONDITIONING_STORE / f"{key}.pt",
        DRIVE_CONDITIONING_STORE / f"{key}.pt") if p.is_file()), None)
    if resolved is None:
        remote = find_blob_anywhere(key)
        if remote is not None:
            CONDITIONING_STORE.mkdir(parents=True, exist_ok=True)
            shutil.copy2(remote, CONDITIONING_STORE / f"{key}.pt")
            resolved = CONDITIONING_STORE / f"{key}.pt"
    if resolved is not None and resolved.parent != CONDITIONING_STORE:
        shutil.copy2(resolved, CONDITIONING_STORE / resolved.name)
        resolved = CONDITIONING_STORE / resolved.name
    return resolved


for shot in SHOTS:
    for gear, key in shot["keys"].items():
        shot["blobs"][gear] = resolve_blob(key)
pending = [(shot, gear) for shot in SHOTS for gear in shot["keys"]
           if shot["blobs"].get(gear) is None]
neg_pending = [v for v, k in NEG_KEYS.items() if resolve_blob(k) is None]
total_blobs = sum(len(s["keys"]) for s in SHOTS)
print(f"conditioning blobs: {total_blobs - len(pending)}/{total_blobs} present, "
      f"{len(pending)} to encode; negative blobs missing: {neg_pending or 'none'}")


# --- shared fetch helper (verbatim semantics from the H3 stage) ---------------
def fetch_verified(expected, subdir):
    """Download one HF file into MODEL_STORE/<subdir>/ with size+sha256 gates."""
    from huggingface_hub import hf_hub_download
    relative_name = f"{subdir}/{expected['hf_filename']}"
    destination = MODEL_STORE / relative_name
    destination.parent.mkdir(parents=True, exist_ok=True)
    record = {"repo": expected["repo"], "file": expected["hf_filename"],
              "expected_bytes": expected["bytes"],
              "expected_gib": round(expected["bytes"] / GIB, 3)}
    if destination.is_file() and destination.stat().st_size == expected["bytes"]:
        record["source"] = "already_on_disk"
        print(f"  reusing {relative_name} ({expected['bytes'] / GIB:.2f} GiB)")
    else:
        free_gib = shutil.disk_usage(str(CONTENT)).free / GIB
        if free_gib < expected["bytes"] / GIB + 3:
            raise RuntimeError(f"Only {free_gib:.1f} GiB free; need "
                               f"{expected['bytes'] / GIB + 3:.1f} GiB for {relative_name}.")
        print(f"  downloading {relative_name} ({expected['bytes'] / GIB:.2f} GiB) ...")
        started = time.perf_counter()
        fetched = hf_hub_download(repo_id=expected["repo"], filename=expected["file"],
                                  local_dir=str(MODEL_STORE / "_hf"), token=False)
        shutil.move(fetched, destination)
        record["source"] = "huggingface"
        record["download_s"] = round(time.perf_counter() - started, 1)
        record["download_mib_per_s"] = round(
            expected["bytes"] / (1024 ** 2) / max(record["download_s"], 1e-6), 1)
        print(f"    {record['download_s']}s ({record['download_mib_per_s']} MiB/s)")
    if destination.stat().st_size != expected["bytes"]:
        raise RuntimeError(
            f"{relative_name} is {destination.stat().st_size} bytes, expected "
            f"{expected['bytes']}. A truncated download surfaces later as a "
            "misleading shape error, so stop here.")
    print("  verifying SHA-256 ...")
    started = time.perf_counter()
    record["sha256"] = sha256_file(destination)
    record["sha256_matches"] = record["sha256"] == expected["sha256"]
    record["verify_s"] = round(time.perf_counter() - started, 1)
    if not record["sha256_matches"]:
        raise RuntimeError(f"{relative_name} SHA-256 mismatch; refusing to continue.")
    print(f"  verified in {record['verify_s']}s")
    link = COMFY_DIR / "models" / relative_name
    link.parent.mkdir(parents=True, exist_ok=True)
    if link.is_symlink():
        if link.resolve() != destination.resolve():
            link.unlink()
            link.symlink_to(destination)
    elif link.exists():
        raise RuntimeError(f"Refusing to replace a non-symlink model path: {link}")
    else:
        link.symlink_to(destination)
    record["linked_at"] = str(link)
    return record


models = {}
print("\nDiT t2v + DiT fun_inp + VAE ...")
models["dit_t2v"] = fetch_verified(DIT_T2V, "diffusion_models")
models["dit_fun_inp"] = fetch_verified(DIT_FUN_INP, "diffusion_models")
models["vae"] = fetch_verified(VIDEO_VAE, "vae")

PROBE["stages"]["models"] = {k: {kk: vv for kk, vv in v.items() if kk != "files"}
                             for k, v in models.items()}
save()
prog("dits+vae ready")


# --- encode every missing blob in one session, then record models -------------

def encode_graph(shot, gear, key_pos, key_neg, image_name):
    """Two branches, both saving (conditioning, latent) blob pairs.

    t2v:     CLIPTextEncode pos/neg + EmptyHunyuanLatentVideo zeros.
    fun_inp: the same text conditioning runs through WanFunInpaintToVideo with
             the start image - it returns (positive, negative, latent) where
             the conditioning now carries the encoded keyframe and the mask
             that pins frame 0 to it.
    """
    geometry = shot["geometries"][gear]
    graph = {
        "2": {"class_type": "CLIPLoader", "inputs": {
            "clip_name": Path(TEXT_ENCODER["hf_filename"]).name, "type": "wan",
            "device": "default"}},
        "11": {"class_type": "CLIPTextEncode", "inputs": {"clip": ["2", 0], "text": shot["prompt"]}},
        "12": {"class_type": "CLIPTextEncode", "inputs": {"clip": ["2", 0], "text": NEGATIVE_PROMPT}},
    }
    if shot.get("has_frame"):
        graph.update({
            "3": {"class_type": "VAELoader", "inputs": {
                "vae_name": Path(VIDEO_VAE["hf_filename"]).name}},
            "5": {"class_type": "LoadImage", "inputs": {"image": image_name}},
            "6": {"class_type": "WanFunInpaintToVideo", "inputs": {
                "positive": ["11", 0], "negative": ["12", 0], "vae": ["3", 0],
                "width": geometry["width"], "height": geometry["height"],
                "length": geometry["length"], "batch_size": 1,
                "start_image": ["5", 0]}},
            "40": {"class_type": "H3SaveConditioning", "inputs": {
                "conditioning": ["6", 0], "samples": ["6", 2],
                "store_dir": str(CONDITIONING_STORE), "key": key_pos}},
            "41": {"class_type": "H3SaveConditioning", "inputs": {
                "conditioning": ["6", 1], "samples": ["6", 2],
                "store_dir": str(CONDITIONING_STORE), "key": key_neg}},
        })
    else:
        graph.update({
            "7": {"class_type": "EmptyHunyuanLatentVideo", "inputs": {
                "width": geometry["width"], "height": geometry["height"],
                "length": geometry["length"], "batch_size": 1}},
            "40": {"class_type": "H3SaveConditioning", "inputs": {
                "conditioning": ["11", 0], "samples": ["7", 0],
                "store_dir": str(CONDITIONING_STORE), "key": key_pos}},
            "41": {"class_type": "H3SaveConditioning", "inputs": {
                "conditioning": ["12", 0], "samples": ["7", 0],
                "store_dir": str(CONDITIONING_STORE), "key": key_neg}},
        })
    return graph


def wait_prompt(base_url, prompt_id, what, timeout_s):
    started = time.perf_counter()
    while time.perf_counter() - started < timeout_s:
        if COMFY_PROCESS is not None and COMFY_PROCESS.poll() is not None:
            raise RuntimeError(f"ComfyUI exited during {what} (check for an OOM kill)")
        try:
            history = requests.get(f"{base_url}/history/{prompt_id}", timeout=60).json()
            if prompt_id in history:
                status = history[prompt_id].get("status", {})
                if status.get("completed") is True or status.get("status_str") == "error":
                    if status.get("status_str") != "success":
                        raise RuntimeError(f"{what} failed: {status}")
                    return round(time.perf_counter() - started, 1)
        except RuntimeError:
            raise
        except Exception:
            pass
        time.sleep(3)
    raise TimeoutError(f"{what} did not finish in {timeout_s}s")


def encode_one(base_url, shot, gear, key_pos, key_neg):
    image_name = f"cond_{key_pos}.jpg"
    (COMFY_DIR / "input").mkdir(parents=True, exist_ok=True)
    if shot.get("has_frame"):
        shutil.copy2(shot["frame_path"], COMFY_DIR / "input" / image_name)
    graph = encode_graph(shot, gear, key_pos, key_neg, image_name)
    prompt_id = str(uuid.uuid4())
    response = requests.post(f"{base_url}/prompt", timeout=120, json={
        "prompt": graph, "client_id": str(uuid.uuid4()), "prompt_id": prompt_id})
    response.raise_for_status()
    if response.json().get("node_errors"):
        raise RuntimeError(f"Node errors: {response.json()['node_errors']}")
    return wait_prompt(base_url, prompt_id, f"encode {key_pos[:18]} gear {gear}", 3600)


if pending or neg_pending:
    print(f"\nDownloading text encoder ({TEXT_ENCODER['bytes'] / GIB:.2f} GiB, one-time) ...")
    models["text_encoder"] = fetch_verified(TEXT_ENCODER, "text_encoders")
    server = start_comfy(port=8190, timeout_s=1200)
    prog("encode server boot")
    try:
        encode_info = requests.get(f"{server}/object_info", timeout=300).json()
        for node in ("CLIPLoader", "CLIPTextEncode", "VAELoader", "LoadImage",
                     "WanFunInpaintToVideo", "EmptyHunyuanLatentVideo",
                     "H3SaveConditioning"):
            if node not in encode_info:
                raise RuntimeError(f"{node} is missing; cannot encode")
        # negative blobs first (shared across shots of the variant), then shots
        plan = []
        for variant in neg_pending:
            plan.append(("negative", variant, NEG_KEYS[variant]))
        for shot, gear in pending:
            plan.append(("shot", shot, gear))
        done = 0
        for kind, obj, arg in plan:
            if kind == "negative":
                variant, key = obj, arg
                stub = {"prompt": NEGATIVE_PROMPT, "has_frame": False,
                        "geometries": {GEAR: GEOMETRY}}
                seconds = encode_one(server, stub, GEAR, key, key)
                CONDITIONING_STORE.mkdir(parents=True, exist_ok=True)
                if not (CONDITIONING_STORE / f"{key}.pt").is_file():
                    raise RuntimeError(f"negative blob {key} was not saved")
                neg_pending = [v for v in neg_pending if v != variant]
                print(f"  negative blob for {variant} saved in {seconds}s")
            else:
                shot, gear = obj, arg
                key_pos = shot["keys"][gear]
                neg = NEG_KEYS[shot["variant"]]
                seconds = encode_one(server, shot, gear, key_pos, neg)
                shot["blobs"][gear] = CONDITIONING_STORE / f"{key_pos}.pt"
                print(f"  [{sum(1 for k, o, a in plan[:done] if k == 'shot') + 1}"
                      f"/{len(pending)}] shot_{shot['num']:02d} gear {gear} "
                      f"({shot['variant']}) encoded in {seconds}s")
            done += 1
            try:
                DRIVE_CONDITIONING_STORE.mkdir(parents=True, exist_ok=True)
                saved_keys = [NEG_KEYS[v] for v in NEG_KEYS
                              if (CONDITIONING_STORE / f"{NEG_KEYS[v]}.pt").is_file()] \
                    + [s["keys"][g] for s in SHOTS for g in s["keys"]
                       if (CONDITIONING_STORE / f"{s['keys'][g]}.pt").is_file()]
                for key in set(saved_keys):
                    source = CONDITIONING_STORE / f"{key}.pt"
                    if source.is_file():
                        shutil.copy2(source, DRIVE_CONDITIONING_STORE / source.name)
            except Exception as exc:
                print(f"  persist failed (non-fatal): {type(exc).__name__}: {exc}")
    finally:
        stop_comfy()
        print("encode session stopped; sampling starts from a clean process")
else:
    print("all conditioning blobs already present")
    models["text_encoder"] = {"status": "SKIPPED_BLOBS_CACHED"}

PROBE["stages"]["models"] = {k: {kk: vv for kk, vv in v.items() if kk != "files"}
                             for k, v in models.items()}
save()
prog("cell3 done (dits+vae ready, blobs encoded)")


In [ ]:
# Stages 3-5 (dual-worker batch) -- TWO ComfyUI servers, one per T4.
#
# The parallelism this kernel exists for, unchanged from the H3/LTX eras: one
# worker thread per T4, each owning a ComfyUI process on its own port, pulling
# from a shared shot queue. What changed with Wan 2.1 1.3B:
#   * the sampling graph loads the conditioning pair from blobs (no encoder),
#     samples 30 uni_pc steps through ModelSamplingSD3 shift 8, and then -
#     in the SAME graph - decodes with the wan VAE (tiled) and muxes a silent
#     MP4 through CreateVideo + SaveVideo. No separate decode stage, no audio
#     VAE: Wan 2.1 is video-only.
#   * the retry ladder shrinks the geometry instead of swapping quants: both
#     DiTs fit a T4, so a smaller checkpoint would buy nothing.
import queue as queue_mod
import threading

KAGGLE_WORKING = Path("/kaggle/working") if Path("/kaggle").is_dir() else WORK
OUT_STORE = KAGGLE_WORKING / "out"
STATE_PATH = KAGGLE_WORKING / "state.json"
OUT_STORE.mkdir(parents=True, exist_ok=True)

SESSION_DEADLINE = time.time() + int(8.4 * 3600)   # margin under the 9h GPU cap
WORKER_PORTS = {0: 8188, 1: 8189}
ENCODE_PORT = 8190           # dedicated blob-encode server (samplers are 8188/8189)
INSURANCE_TOKEN = "KGAT_abba3bba83ed8f6a11855f7a819228ea"
JOBS_DS = "freedomform/rmc-studio-jobs"
OUTPUTS_DS = "freedomform/rmc-studio-outputs"
WORKER_POLL_S = 25           # cheap jobs-dataset poll cadence (no GPU cost)
WORKER_IDLE_TIMEOUT_S = 1800  # exit after 30 idle minutes to save GPU quota
import zlib as _zlib

PUSH_LOCK = threading.Lock()
def push_insurance(tag):
    """serialized wrapper: feeder and both workers push the same dataset"""
    with PUSH_LOCK:
        return _push_insurance_impl(tag)

state = (json.loads(STATE_PATH.read_text(encoding="utf-8"))
         if STATE_PATH.exists() else {"shots": {}})
state.setdefault("shots", {})
prog("cell4 runtime init")

def save_state():
    STATE_PATH.write_text(json.dumps(state, indent=1, ensure_ascii=False),
                          encoding="utf-8")

save_state()

# --- per-worker ComfyUI process control ---------------------------------------
WORKER_TABLE = {}

def stop_comfy_port(port, term_timeout_s=15):
    entry = WORKER_TABLE.pop(port, None)
    if entry is None:
        return
    process = entry["process"]
    if process.poll() is None:
        try:
            os.killpg(process.pid, signal.SIGTERM)
            process.wait(timeout=term_timeout_s)
        except (subprocess.TimeoutExpired, ProcessLookupError):
            try:
                os.killpg(process.pid, signal.SIGKILL)
                process.wait(timeout=15)
            except (ProcessLookupError, subprocess.TimeoutExpired):
                pass
    try:
        entry["log_handle"].close()
    except Exception:
        pass

def spawn_comfy(port, gpu, dtype_flag=None, timeout_s=900):
    """One ComfyUI process pinned to one GPU. dtype_flag is unused for Wan
    (comfy picks fp16 on sm75 by itself); kept for signature compatibility."""
    stop_comfy_port(port)
    log_path = WORK / f"comfy_p{port}.log"
    log_handle = log_path.open("a", encoding="utf-8", buffering=1)
    supported = supported_cli_flags()
    command = [sys.executable, "main.py", "--listen", "127.0.0.1", "--port", str(port),
               "--disable-auto-launch", "--preview-method", "none", "--cache-classic"]
    requested, applied, dropped = [], [], []
    if dtype_flag:
        requested.append(dtype_flag)
    for flag in requested:
        (applied if flag in supported else dropped).append(flag)
        if flag in supported:
            command.append(flag)
    for forbidden in FORBIDDEN_FLAGS:
        if forbidden in command:
            raise RuntimeError(f"{forbidden} is forbidden here.")
    env = scrubbed_child_environment({"CUDA_VISIBLE_DEVICES": str(gpu),
                                      "PYTHONUNBUFFERED": "1"})
    print(f"$ CUDA_VISIBLE_DEVICES={gpu} " + " ".join(command))
    process = subprocess.Popen(command, cwd=COMFY_DIR, env=env, stdout=log_handle,
                               stderr=subprocess.STDOUT, start_new_session=True)
    base = f"http://127.0.0.1:{port}"
    deadline = time.time() + timeout_s
    while time.time() < deadline:
        if process.poll() is not None:
            raise RuntimeError(f"ComfyUI (port {port}) exited during startup:\n"
                               + log_path.read_text(errors="replace")[-4000:])
        try:
            if requests.get(f"{base}/system_stats", timeout=5).status_code == 200:
                WORKER_TABLE[port] = {"process": process, "log": log_path,
                                      "log_handle": log_handle, "base_url": base}
                return WORKER_TABLE[port]
        except Exception:
            pass
        time.sleep(2)
    stop_comfy_port(port)
    raise RuntimeError(f"ComfyUI startup timeout on port {port}")

def server_log_slice(port, offset):
    entry = WORKER_TABLE.get(port)
    if entry is not None and entry["log_handle"] is not None:
        entry["log_handle"].flush()
    path = WORK / f"comfy_p{port}.log"
    if not path.exists():
        return ""
    with path.open("rb") as reader:
        reader.seek(offset)
        return reader.read().decode("utf-8", errors="replace")

def dropdown_from(info, node_type, input_name):
    inputs = info.get(node_type, {}).get("input", {})
    spec = (inputs.get("required", {}) | inputs.get("optional", {})).get(input_name)
    return spec[0] if spec and isinstance(spec[0], list) else []

# --- the sample+decode graph (per-shot; no text encoder) ------------------------
GUARD_HEARTBEAT_S = 30

def build_sample_graph(shot, gear, seed, latent_key, negative_blob_key):
    """One roundtrip: load blobs -> 30-step uni_pc sample -> tiled VAE decode ->
    silent mp4. The negative blob's latent is ignored on purpose - the sampler
    seeds from the positive blob's zeros/Fun-InP latent (node 2 output 1)."""
    dit_name = shot["dit"]["hf_filename"]
    video_prefix = f"wan13_video/job_{shot['job_id']}/{latent_key}"
    return {
        "1": {"class_type": "UNETLoader", "inputs": {"unet_name": dit_name,
                                                     "weight_dtype": "default"}},
        "2": {"class_type": "H3LoadConditioning", "inputs": {
            "store_dir": str(CONDITIONING_STORE), "key": shot["keys"][gear]}},
        "3": {"class_type": "H3LoadConditioning", "inputs": {
            "store_dir": str(CONDITIONING_STORE), "key": negative_blob_key}},
        "4": {"class_type": "ModelSamplingSD3", "inputs": {
            "model": ["1", 0], "shift": SHIFT}},
        "5": {"class_type": "KSampler", "inputs": {
            "model": ["4", 0], "positive": ["2", 0], "negative": ["3", 0],
            "latent_image": ["2", 1], "seed": seed, "steps": STEPS, "cfg": CFG,
            "sampler_name": SAMPLER_NAME, "scheduler": SCHEDULER, "denoise": 1.0}},
        "6": {"class_type": "H3SaveLatent", "inputs": {
            "samples": ["5", 0], "store_dir": str(LATENT_STORE), "key": latent_key}},
        "7": {"class_type": "H3LatentProbe", "inputs": {
            "samples": ["6", 0], "filename_prefix": f"wan13/job_{shot['job_id']}"}},
        "8": {"class_type": "VAELoader", "inputs": {
            "vae_name": Path(VIDEO_VAE["hf_filename"]).name}},
        "9": {"class_type": "VAEDecodeTiled", "inputs": {
            "samples": ["5", 0], "vae": ["8", 0], "tile_size": 512, "overlap": 64,
            "temporal_size": 64, "temporal_overlap": 16}},
        "10": {"class_type": "CreateVideo", "inputs": {
            "images": ["9", 0], "fps": float(FPS)}},
        "11": {"class_type": "SaveVideo", "inputs": {
            "video": ["10", 0], "filename_prefix": video_prefix, "format": "auto"}},
    }

def validate(graph, object_info, port):
    errors = []
    serialized = json.dumps(graph)
    if "CLIPLoader" in serialized:
        errors.append("CLIPLoader present: the 6.27 GiB text encoder must not load here")
    for node_id, spec in graph.items():
        info = object_info.get(spec["class_type"])
        if info is None:
            errors.append(f"{node_id}: unknown class {spec['class_type']}")
            continue
        schema = info.get("input", {})
        allowed = set(schema.get("required", {})) | set(schema.get("optional", {})) \
            | set(schema.get("hidden", {}))
        supplied = set(spec["inputs"])
        if set(schema.get("required", {})) - supplied:
            errors.append(f"{node_id}/{spec['class_type']} missing "
                          f"{sorted(set(schema.get('required', {})) - supplied)}")
        if supplied - allowed:
            errors.append(f"{node_id}/{spec['class_type']} unknown "
                          f"{sorted(supplied - allowed)}")
    if errors:
        raise RuntimeError(f"Sample graph validation failed (port {port}):\n"
                           + "\n".join(errors))

# --- progress parsing and the budget guard -------------------------------------
ANSI_RE = re.compile(r"\x1b\[[0-9;]*m")
TQDM_RE = re.compile(
    r"(\d+)%\|[^|\n]*\|\s*(\d+)/(\d+)\s*\[([0-9:]+)<([0-9:]+|\?)"
    r"(?:,\s*([0-9.]+)(s/it|it/s))?")

CLIFF_PROBE_WINDOW_S = 120
CLIFF_PROBE_MAX_WINDOWS = 2
CLIFF_PROJECTED_BUDGET_S = 3600
SAMPLER_START_TIMEOUT_S = 1500
VIDEO_INSPECT_EVERY = 8

def parse_clock(value):
    total = 0.0
    for part in value.split(":"):
        total = total * 60 + float(part)
    return total

def latest_progress(text):
    matches = list(TQDM_RE.finditer(ANSI_RE.sub("", text).replace("\r", "\n")))
    if not matches:
        return None
    percent, done, total, elapsed, _remaining, rate, unit = matches[-1].groups()
    done, elapsed_s = int(done), parse_clock(elapsed)
    return {"percent": int(percent), "steps_completed": done,
            "steps_total": int(total), "elapsed_in_loop_s": elapsed_s,
            "observed_s_per_step": (elapsed_s / done) if done else None,
            "reported_s_per_step": (float(rate) if unit == "s/it"
                                    else (1.0 / float(rate) if rate and float(rate) else None))
            if rate else None}

def new_guard_runtime(started):
    return {"started": started, "sampler_started_at": None, "windows_used": 0,
            "last_heartbeat": started, "settled": False, "previous_window": None}

def guard_decide(progress, now, runtime, guard_state, steps, budget_s):
    """Times two probe windows of the sampler, projects the total denoise time
    from the observed s/step, and cancels projections that overrun the budget."""
    runtime = dict(runtime)
    messages = []
    waited = now - runtime["started"]
    if waited < 0 or waited > 86400:
        raise RuntimeError(f"Guard clock is inconsistent: waited={waited}.")
    if progress is None:
        if now - runtime["last_heartbeat"] >= GUARD_HEARTBEAT_S:
            runtime["last_heartbeat"] = now
            messages.append(f"[guard] {waited:.0f}s: loading, no sampler progress yet "
                            f"(timeout {SAMPLER_START_TIMEOUT_S}s)")
        if waited > SAMPLER_START_TIMEOUT_S:
            guard_state["seconds_waited_for_sampler"] = round(waited, 1)
            return ({"stop": True, "outcome": "SAMPLER_NEVER_STARTED",
                     "messages": messages}, runtime)
        return ({"stop": False, "outcome": None, "messages": messages}, runtime)
    if runtime["sampler_started_at"] is None:
        runtime["sampler_started_at"] = now - progress["elapsed_in_loop_s"]
        guard_state["sampler_started"] = True
        guard_state["seconds_to_sampler_start"] = round(
            runtime["sampler_started_at"] - runtime["started"], 1)
        messages.append(f"[guard] sampler started after "
                        f"{guard_state['seconds_to_sampler_start']}s")
    if runtime["settled"]:
        return ({"stop": False, "outcome": "WITHIN_BUDGET", "messages": messages}, runtime)
    observed = now - runtime["sampler_started_at"]
    if progress["steps_completed"] <= 0:
        no_step_deadline = max(CLIFF_PROBE_WINDOW_S * CLIFF_PROBE_MAX_WINDOWS,
                               2.0 * budget_s / max(steps, 1))
        guard_state["no_step_deadline_s"] = round(no_step_deadline, 1)
        if observed >= no_step_deadline:
            guard_state["observation_s"] = round(observed, 1)
            guard_state["steps_completed_at_decision"] = 0
            messages.append(f"[guard] {observed:.0f}s observed and not one step has "
                            f"finished (deadline {no_step_deadline:.0f}s)")
            return ({"stop": True, "outcome": "CLIFF_NO_STEP_COMPLETED",
                     "messages": messages}, runtime)
    if observed < CLIFF_PROBE_WINDOW_S * (runtime["windows_used"] + 1):
        return ({"stop": False, "outcome": None, "messages": messages}, runtime)
    runtime["windows_used"] += 1
    marginal = None
    previous = runtime["previous_window"]
    if previous and progress["steps_completed"] > previous["steps_completed"]:
        marginal = ((progress["elapsed_in_loop_s"] - previous["elapsed_in_loop_s"])
                    / (progress["steps_completed"] - previous["steps_completed"]))
    runtime["previous_window"] = {
        "elapsed_in_loop_s": progress["elapsed_in_loop_s"],
        "steps_completed": progress["steps_completed"]}
    guard_state.update({"windows_used": runtime["windows_used"],
                        "observation_s": round(observed, 1),
                        "steps_completed_at_decision": progress["steps_completed"],
                        "observed_s_per_step": progress["observed_s_per_step"],
                        "marginal_s_per_step": round(marginal, 2) if marginal else None})
    if progress["steps_completed"] <= 0:
        messages.append(f"[guard] {observed:.0f}s observed, still 0 steps "
                        f"(cancel at {guard_state.get('no_step_deadline_s')}s)")
        return ({"stop": False, "outcome": None, "messages": messages}, runtime)
    projected = progress["observed_s_per_step"] * steps
    guard_state["projected_denoise_total_s"] = round(projected, 1)
    messages.append(f"[guard] {progress['steps_completed']}/{steps} steps in "
                    f"{observed:.0f}s -> {progress['observed_s_per_step']:.1f} s/step, "
                    f"projected {projected / 60:.1f} min")
    if projected > budget_s:
        return ({"stop": True, "outcome": "CLIFF_PROJECTED_OVER_BUDGET",
                 "messages": messages}, runtime)
    runtime["settled"] = True
    guard_state["outcome"] = "WITHIN_BUDGET"
    return ({"stop": False, "outcome": "WITHIN_BUDGET", "messages": messages}, runtime)

def cancel_prompt(port, prompt_id):
    outcome = {"cancel_requested": False, "drained": False}
    base = f"http://127.0.0.1:{port}"
    try:
        response = requests.post(f"{base}/api/jobs/{prompt_id}/cancel", timeout=30)
        response.raise_for_status()
        outcome["cancel_requested"] = bool(response.json().get("cancelled"))
    except Exception as exc:
        outcome["cancel_error"] = type(exc).__name__
    deadline = time.time() + 120
    while time.time() < deadline:
        try:
            queue = requests.get(f"{base}/queue", timeout=30).json()
            active = {item[1] for item in queue.get("queue_running", [])
                      + queue.get("queue_pending", [])
                      if isinstance(item, list) and len(item) > 1}
            if prompt_id not in active:
                outcome["drained"] = True
                return outcome
        except Exception:
            pass
        time.sleep(2)
    return outcome

# --- sampling + decoding one attempt --------------------------------------------
def generate_one(port, shot, gear, seed, latent_key, budget_s, record):
    """Runs the whole sample+decode graph in one fresh ComfyUI process and
    returns the produced mp4 path, or None (record explains)."""
    gpu = 0 if port == WORKER_PORTS[0] else 1
    server = spawn_comfy(port, gpu)
    base = server["base_url"]
    log_offset = server["log"].stat().st_size if server["log"].exists() else 0
    try:
        info = requests.get(f"{base}/object_info", timeout=300).json()
        required = {"UNETLoader", "H3LoadConditioning", "ModelSamplingSD3", "KSampler",
                    "H3SaveLatent", "H3LatentProbe", "VAELoader", "VAEDecodeTiled",
                    "CreateVideo", "SaveVideo"}
        missing = sorted(required - set(info))
        if missing:
            raise RuntimeError(f"Missing required nodes: {missing}")
        if shot["dit"]["hf_filename"] not in dropdown_from(info, "UNETLoader", "unet_name"):
            raise RuntimeError(f"DiT {shot['dit']['hf_filename']} not visible to UNETLoader")
        if Path(VIDEO_VAE["hf_filename"]).name not in dropdown_from(info, "VAELoader", "vae_name"):
            raise RuntimeError("wan VAE not visible to VAELoader")
        graph = build_sample_graph(shot, gear, seed, latent_key,
                                   NEG_KEYS[shot["variant"]])
        validate(graph, info, port)
        prompt_id = str(uuid.uuid4())
        response = requests.post(f"{base}/prompt", timeout=120, json={
            "prompt": graph, "client_id": str(uuid.uuid4()), "prompt_id": prompt_id})
        response.raise_for_status()
        queued = response.json()
        if queued.get("node_errors"):
            raise RuntimeError(f"Node errors: {queued['node_errors']}")
        started = time.perf_counter()
        print(f"shot_{shot['num']:02d}: sampling {shot['geometries'][gear]['length']}f "
              f"gear {gear} seed {seed} ({shot['dit']['hf_filename']}) on port {port}")
        shown = log_offset
        guard_runtime = new_guard_runtime(started)
        guard_state = {"outcome": None, "sampler_started": False}
        record_out = None
        deadline = min(SESSION_DEADLINE,
                       time.time() + SAMPLER_START_TIMEOUT_S + budget_s + 1800)
        while time.time() < deadline:
            if server["process"].poll() is not None:
                raise RuntimeError("ComfyUI exited during generation (check for an OOM kill)")
            chunk = server_log_slice(port, shown)
            shown += len(chunk.encode("utf-8"))
            for line in chunk.splitlines():
                if line.strip() and any(t in line for t in (
                        "error", "Error", "out of memory", "Requested to load",
                        "Patched", "lora key")):
                    print("    " + line[:160])
            try:
                history = requests.get(f"{base}/history/{prompt_id}", timeout=60).json()
                if prompt_id in history:
                    status = history[prompt_id].get("status", {})
                    if status.get("completed") is True or status.get("status_str") == "error":
                        record_out = history[prompt_id]
                        break
            except Exception:
                pass
            progress = latest_progress(server_log_slice(port, log_offset))
            now = time.perf_counter()
            decision, guard_runtime = guard_decide(progress, now, guard_runtime,
                                                   guard_state, STEPS, budget_s)
            for message in decision["messages"]:
                print("    " + message)
            if decision["stop"]:
                guard_state["outcome"] = decision["outcome"]
                guard_state["cancellation"] = cancel_prompt(port, prompt_id)
                break
            time.sleep(5)
        elapsed = round(time.perf_counter() - started, 1)
        outcome = guard_state.get("outcome")
        record.update({"generate_elapsed_s": elapsed, "guard_state": guard_state})
        if outcome in ("CLIFF_NO_STEP_COMPLETED", "CLIFF_PROJECTED_OVER_BUDGET",
                       "SAMPLER_NEVER_STARTED"):
            record["sample_status"] = "CLIFF"
            record["failure_class"] = outcome
            return None
        if record_out is None:
            record["sample_status"] = "NO_HISTORY"
            return None
        status = record_out.get("status", {})
        log_text = server_log_slice(port, log_offset)
        record["oom_seen"] = bool(re.search(
            r"CUDA out of memory|torch\.OutOfMemoryError", log_text, re.I))
        if status.get("status_str") != "success":
            record["sample_status"] = "FAILED"
            record["history_status_str"] = status.get("status_str")
            record["history_messages"] = [str(m)[:300] for m in (status.get("messages") or [])][:8]
            return None
        latent_path = LATENT_STORE / f"{latent_key}.pt"
        if not latent_path.is_file():
            record["sample_status"] = "NO_LATENT"
            return None
        summaries = sorted((OUTPUT_DIR / "wan13").glob(
            f"job_{shot['job_id']}*.latent_summary.json"), key=lambda p: p.stat().st_mtime)
        if summaries:
            summary = json.loads(summaries[-1].read_text(encoding="utf-8"))
            record["latent_summary"] = {"shapes": summary.get("shapes"),
                                        "finite": summary.get("finite")}
            finite = (summary.get("finite") or [])
            if finite and not all(finite):
                record["sample_status"] = "NON_FINITE_LATENT"
                return None
        produced = []
        for node_output in (record_out.get("outputs") or {}).values():
            for entry in node_output.get("images", []) + node_output.get("videos", []) \
                    + node_output.get("gifs", []):
                if isinstance(entry, dict) and entry.get("filename"):
                    produced.append(OUTPUT_DIR / entry.get("subfolder", "") / entry["filename"])
        if not produced:
            directory = OUTPUT_DIR / "wan13_video" / shot_num_dir(video_prefix)
            produced = sorted(directory.glob("**/*"), key=lambda p: p.stat().st_mtime) \
                if directory.is_dir() else []
        produced = [p for p in produced if p.is_file() and p.suffix in (".mp4", ".webm", ".mkv")]
        if not produced:
            record["sample_status"] = "NO_FILE"
            return None
        record["sample_status"] = "SUCCESS"
        print(f"shot_{shot['num']:02d}: generated ok in {elapsed}s "
              f"({guard_state.get('observed_s_per_step')} s/step)")
        return produced[-1]
    finally:
        stop_comfy_port(port)

def shot_num_dir(video_prefix):
    return str(Path(video_prefix).parent.name)

# --- is there actually a picture in it? (PyAV QA, silent mp4 expected) ----------
def inspect_video(mp4, record):
    inspection = {}
    try:
        import av
        with av.open(str(mp4)) as container:
            streams = {s.type: s for s in container.streams}
            inspection["streams"] = sorted(streams)
            vstream = streams.get("video")
            if vstream is not None:
                inspection["video"] = {
                    "codec": vstream.codec_context.name,
                    "width": vstream.codec_context.width,
                    "height": vstream.codec_context.height,
                    "frames": vstream.frames,
                    "avg_rate": float(vstream.average_rate)
                    if vstream.average_rate else None}
            if "audio" in streams:
                inspection["unexpected_audio"] = True
        with av.open(str(mp4)) as container:
            brightness, index = [], 0
            for frame in container.decode(video=0):
                if index % VIDEO_INSPECT_EVERY == 0:
                    plane = frame.to_ndarray(format="gray")
                    brightness.append((index, float(plane.mean()), float(plane.std())))
                index += 1
            inspection["frames_decoded"] = index
            if brightness:
                means = [m for _, m, _ in brightness]
                stds = [s for _, _, s in brightness]
                inspection["mean_luma"] = round(sum(means) / len(means), 2)
                inspection["mean_std"] = round(sum(stds) / len(stds), 2)
                inspection["near_black_frames"] = sum(
                    1 for m, s in zip(means, stds) if m < 4.0 and s < 2.0)
                inspection["frames_sampled"] = len(means)
        record["inspection"] = inspection
    except Exception as exc:
        inspection["error"] = f"{type(exc).__name__}: {exc}"
        record["inspection"] = inspection
    return inspection

# --- insurance: push partial output to a dataset (v3's proven pattern) ----------
def _push_insurance_impl(tag):
    try:
        os.environ["KAGGLE_API_TOKEN"] = INSURANCE_TOKEN
        (OUT_STORE / "state.json").write_text(
            json.dumps(state, indent=1, ensure_ascii=False), encoding="utf-8")
        meta_path = OUT_STORE / "dataset-metadata.json"
        if not meta_path.exists():
            json.dump({"title": "rmc-studio-outputs", "id": "freedomform/rmc-studio-outputs",
                       "licenses": [{"name": "CC0-1.0"}]},
                      open(meta_path, "w"))
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kaggle"],
                       capture_output=True, timeout=300)
        # create first (no-op if it already exists), then push a version --
        # v3's lesson: `datasets version` alone fails while the dataset is absent.
        created = subprocess.run(
            [sys.executable, "-m", "kaggle", "datasets", "create", "-p", str(OUT_STORE),
             "--dir-mode", "zip"], capture_output=True, text=True, timeout=1800)
        blob = (created.stdout or "") + (created.stderr or "")
        if "successfully created" not in blob.lower():
            version = subprocess.run(
                [sys.executable, "-m", "kaggle", "datasets", "version",
                 "-p", str(OUT_STORE), "--dir-mode", "zip",
                 "-m", f"{tag} {time.strftime('%H:%M')}"],
                capture_output=True, text=True, timeout=1800)
            print(f"insurance version rc={version.returncode} "
                  f"{(version.stdout or version.stderr)[-160:]}")
        else:
            print("insurance dataset created (first push done)")
    except Exception as exc:
        print(f"insurance push failed (non-fatal): {type(exc).__name__}: {exc}")

# --- one shot through the ladder -------------------------------------------------
def run_shot(port, shot, record):
    """(primary gear, seed) -> (primary gear, seed+1000) -> (fallback gear,
    seed+1000). Same DiT on every rung: both Wan 1.3B variants fit a T4, so
    there is no quant to fall back to - only geometry and noise change."""
    attempts = [
        (shot["gear_primary"], shot["seed"]),
        (shot["gear_primary"], shot["seed"] + 1000),
        (shot["gear_fallback"], shot["seed"] + 1000),
    ]
    for attempt_index, (gear, seed) in enumerate(attempts):
        key = shot["keys"].get(gear)
        if key is None or not (CONDITIONING_STORE / f"{key}.pt").is_file():
            print(f"shot_{shot['num']:02d}: no blob for gear {gear}, skipping attempt")
            continue
        frames = shot["geometries"][gear]["length"]
        budget_s = int(CLIFF_PROJECTED_BUDGET_S * max(1.0, frames / 81.0) ** 1.6)
        latent_key = f"job_{shot['job_id']}_g{gear.replace('.', '')}_s{seed}"
        record.update({"attempt": attempt_index, "gear": gear, "seed": seed,
                       "dit": shot["dit"]["hf_filename"], "frames": frames,
                       "budget_s": budget_s})
        mp4 = generate_one(port, shot, gear, seed, latent_key, budget_s, record)
        if mp4 is None:
            print(f"shot_{shot['num']:02d}: attempt {attempt_index + 1} -> "
                  f"{record.get('sample_status')}, trying next ladder rung")
            continue
        inspection = inspect_video(mp4, record)
        record["mp4"] = str(mp4)
        return mp4
    return None

# --- the two workers -------------------------------------------------------------
# --- [studio-worker] persistent runtime (feeder owns ALL intake) -----------------
QUEUE = queue_mod.Queue()
QUEUE_LOCK = threading.Lock()
state_lock = threading.Lock()
OK_COUNT = {"n": 0}
ACTIVE = {"n": 0}
LAST_WORK = {"t": time.time()}
SHUTDOWN = threading.Event()
SEEN_JOBS = set()

def estimated_shot_seconds(shot):
    """encode ~150 + sample ~30 steps x ~8 s/step scaled by frames + decode/io."""
    frames = float(shot.get("frames", 81))
    return 150 + 360 * (frames / 81.0) ** 1.6 + 120

def worker(gpu):
    port = WORKER_PORTS[gpu]
    while True:
        with QUEUE_LOCK:
            if QUEUE.empty():
                if SHUTDOWN.is_set() or time.time() - LAST_WORK["t"] > WORKER_IDLE_TIMEOUT_S:
                    return
                time.sleep(5)
                continue
            shot = QUEUE.get()
            LAST_WORK["t"] = time.time()
        try:
            done_state = state["shots"].get(shot["job_id"], {})
            if done_state.get("status") == "done" \
                    and (OUT_STORE / f"job_{shot['job_id']}.mp4").is_file():
                print(f"shot_{shot['num']:02d}: already done, skipping (resume)")
                continue
            remaining = SESSION_DEADLINE - time.time()
            estimate = estimated_shot_seconds(shot)
            if remaining < estimate:
                print(f"shot_{shot['num']:02d}: needs ~{estimate / 60:.0f} min, "
                      f"{remaining / 60:.0f} min left -> skipped_time")
                with state_lock:
                    state["shots"][shot["job_id"]] = {
                        "status": "skipped_time", "gpu": gpu}
                    save_state()
                continue
            ACTIVE["n"] += 1
            record = {"gpu": gpu, "started_at": utc_now(), "variant": shot["variant"]}
            started = time.perf_counter()
            mp4 = run_shot(port, shot, record)
            prog(f"shot {shot['num']} run_shot returned gpu{gpu} "
                 f"status={record.get('sample_status', '?')}")
            record["elapsed_s"] = round(time.perf_counter() - started, 1)
            with state_lock:
                if mp4 is not None:
                    dest = OUT_STORE / f"job_{shot['job_id']}.mp4"
                    shutil.copy2(mp4, dest)
                    record.update({"status": "done", "file": str(dest),
                                   "bytes": dest.stat().st_size})
                    OK_COUNT["n"] += 1
                    # push after EVERY finished job (the site polls the dataset)
                    threading.Thread(target=push_insurance,
                                     args=(f"{OK_COUNT['n']} shots",),
                                     daemon=True).start()
                else:
                    record["status"] = "failed"
                state["shots"][shot["job_id"]] = record
                save_state()
        except Exception as exc:
            with state_lock:
                state["shots"][shot["job_id"]] = {
                    "status": "error", "gpu": gpu,
                    "error": f"{type(exc).__name__}: {exc}"[:600]}
                save_state()
            print(f"shot_{shot['num']:02d}: EXCEPTION {type(exc).__name__}: {exc}")
        finally:
            stop_comfy_port(port)
            ACTIVE["n"] = max(0, ACTIVE["n"] - 1)
            LAST_WORK["t"] = time.time()
            QUEUE.task_done()

# --- [studio-worker] feeder: Kaggle-API intake -> prepare -> encode -> queue -----
def kaggle_cli(*args, timeout=900):
    os.environ["KAGGLE_API_TOKEN"] = INSURANCE_TOKEN
    return subprocess.run([sys.executable, "-m", "kaggle", *args],
                          capture_output=True, text=True, timeout=timeout)

def ensure_kaggle_cli():
    """the preinstalled 1.5.x CLI on Kaggle images cannot authenticate with a
    KAGGLE_API_TOKEN (KGAT_...) -- upgrade to the 1.6+/1.7.x line that can"""
    try:
        r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "kaggle"],
                           capture_output=True, text=True, timeout=600)
        print(f"[cli] kaggle upgrade rc={r.returncode}", flush=True)
    except Exception as exc:
        print(f"[cli] kaggle upgrade failed: {type(exc).__name__}: {exc}", flush=True)

def jobs_line():
    try:
        r = kaggle_cli("datasets", "list", "--mine", "--search", JOBS_DS, timeout=120)
        for line in r.stdout.splitlines():
            if line.startswith(JOBS_DS):
                return line
        print(f"[list] no match rc={r.returncode} out={r.stdout[:200]!r} "
              f"err={r.stderr[:300]!r}", flush=True)
    except Exception as exc:
        print(f"[feeder] jobs_line failed: {type(exc).__name__}: {exc}", flush=True)
    return ""

def wait_jobs_ready(timeout_s=420):
    """tolerant ready-gate -- a just-pushed version takes a while to process"""
    t0 = time.time()
    while time.time() - t0 < timeout_s:
        try:
            r = kaggle_cli("datasets", "status", JOBS_DS, timeout=120)
            out = (r.stdout or "").strip().split("\n")[0].strip().lower()
            print(f"[feeder/status] rc={r.returncode} out={out[:60]!r}", flush=True)
            if "processing" not in out and "pending" not in out \
                    and "queued" not in out:
                return True
        except Exception as exc:
            print(f"[feeder/status] error {type(exc).__name__}: {exc}", flush=True)
        time.sleep(20)
    return False

def download_jobs(dest):
    shutil.rmtree(dest, ignore_errors=True)
    dest.mkdir(parents=True, exist_ok=True)
    try:
        kaggle_cli("datasets", "download", JOBS_DS, "-p", str(dest), "--unzip")
    except Exception as exc:
        print(f"[feeder] jobs download failed: {type(exc).__name__}: {exc}")
        return None
    found = sorted(dest.glob("**/shots.json"))
    return found[0].parent if found else None

def prepare_and_encode(entry, base_dir):
    """the cell-3 SHOTS construction + blob encode, for post-startup arrivals."""
    frame_path = base_dir / entry["frame"]
    if not frame_path.is_file():
        raise RuntimeError(f"keyframe missing: {frame_path}")
    variant = variant_for(entry)
    gears = sorted({entry.get("gear_primary", GEAR), entry.get("gear_fallback", "0.2")})
    geometries = {g: geometry_for(g, int(entry["frames"])) for g in gears}
    jid = str(entry["job_id"])
    frame_sha = hashlib.sha256(frame_path.read_bytes()).hexdigest() \
        if variant == "fun_inp" else None
    shot = {
        "num": int(entry["num"]), "job_id": jid, "seconds": entry["seconds"],
        "frames": int(entry["frames"]), "prompt": entry["prompt"],
        "frame_path": frame_path, "has_frame": variant == "fun_inp",
        "variant": variant, "dit": dit_for(variant),
        "gear_primary": entry.get("gear_primary", GEAR),
        "gear_fallback": entry.get("gear_fallback", "0.2"),
        "geometries": geometries,
        "keys": {g: conditioning_key_for(entry["prompt"], frame_sha, gm, variant)
                 for g, gm in geometries.items()},
        "seed": 42 + _zlib.crc32(jid.encode("utf-8")) % 100000,
        "blobs": {},
    }
    for g, key in shot["keys"].items():
        shot["blobs"][g] = resolve_blob(key)
    todo = [(g, k) for g, k in shot["keys"].items() if shot["blobs"].get(g) is None]
    neg_missing = resolve_blob(NEG_KEYS[variant]) is None
    if todo or neg_missing:
        if not (MODEL_STORE / "text_encoders" / TEXT_ENCODER["hf_filename"]).is_file():
            print("[worker] downloading text encoder (one-time 6.27 GiB) ...")
            fetch_verified(TEXT_ENCODER, "text_encoders")
        server = spawn_comfy(ENCODE_PORT, 0)   # encode holds no DiT
        prog("encode server boot")
        try:
            if neg_missing:
                stub = {"prompt": NEGATIVE_PROMPT, "has_frame": False,
                        "geometries": {GEAR: GEOMETRY}}
                encode_one(server["base_url"], stub, GEAR,
                           NEG_KEYS[variant], NEG_KEYS[variant])
            for g, key in todo:
                shot["blobs"][g] = CONDITIONING_STORE / f"{key}.pt"
                encode_one(server["base_url"], shot, g, key, NEG_KEYS[variant])
                prog(f"shot {shot['num']} gear {g} encoded")
        finally:
            stop_comfy_port(ENCODE_PORT)
    return shot

def render_test_stub(entry, base_dir):
    """cheap placeholder clip for test batches (no GPU work at all; silent
    832x480@16 like the real Wan output)."""
    jid = str(entry["job_id"])
    sec = int(entry["seconds"])
    dest = OUT_STORE / f"job_{jid}.mp4"
    cmd = ["ffmpeg", "-y", "-v", "error",
           "-f", "lavfi", "-i", f"color=c=0x1f2937:s=832x480:r=16:d={sec}",
           "-vf", ("drawtext=text='TEST {jid}':fontcolor=white@0.85:"
                   "x=(w-tw)/2:y=(h-th)/2:fontsize=28"),
           "-c:v", "libx264", "-preset", "veryfast", "-crf", "28",
           "-movflags", "+faststart",
           str(dest)]
    r = subprocess.run(cmd, capture_output=True, text=True)
    assert r.returncode == 0, f"ffmpeg failed for job {jid}: {r.stderr[-500:]}"
    return dest

# --- [vebai] stateful terminal + merge + telegram mirror -----------------------
# Shared code embedded into BOTH Kaggle workers (GPU real + CPU test).
# Provides:
#   * PersistentShell  - a real long-lived bash per terminal_id: cd, exports,
#                        venv activation and background processes survive
#                        between commands; timeout kills the process group and
#                        respawns the shell (cwd is preserved).
#   * handle_special() - manifest entries with mode 'cmd' / 'merge'.
#   * Telegram mirror  - workspace snapshot (tar.gz) + files produced during a
#                        command are uploaded to the owner channel with the
#                        terminal id in the caption; latest snapshot file_id is
#                        returned to the server (cmd_<job>.txt) so the next
#                        worker session can restore the workspace.
import json
import os
import subprocess
import time
from pathlib import Path

import re as _re
import selectors as _selectors
import signal as _signal
import tarfile as _tarfile
import uuid as _uuid

TERM_ROOT = Path("/kaggle/working/terminals")
try:
    TERM_ROOT.mkdir(parents=True, exist_ok=True)
except Exception:  # non-Kaggle environment (local tests): fall back to /tmp
    TERM_ROOT = Path("/tmp/vebai_terminals")
    TERM_ROOT.mkdir(parents=True, exist_ok=True)
TG = {"token": None, "chat_id": None, "restored": False}
SHELLS = {}
_LAST_SNAP = {}   # terminal_id -> ((files, bytes), file_id)


def _requests():
    import requests
    return requests


def _load_telegram_cfg(base):
    """lazy read of config.json shipped with the jobs batch; re-reads on every
    batch until token+chat_id are known (the first batch can race the server's
    chat_id discovery). Workspace snapshot restore happens once per session."""
    need_cfg = not (TG["token"] and TG["chat_id"])
    if not need_cfg and TG["restored"]:
        return
    try:
        cfg = json.loads((Path(base) / "config.json").read_text(encoding="utf-8"))
        if not TG["token"]:
            TG["token"] = cfg.get("telegram_token") or None
        if not TG["chat_id"] and cfg.get("telegram_chat_id"):
            TG["chat_id"] = str(cfg.get("telegram_chat_id"))
        snaps = cfg.get("term_snapshots") or {}
        if TG["restored"] or not (TG["token"] and TG["chat_id"]):
            return
        TG["restored"] = True
        if TG["token"] and TG["chat_id"]:
            for tid, fid in snaps.items():
                try:
                    ws = _term_ws(tid)
                    if any(ws.iterdir()):
                        continue  # session already has this workspace
                    dest = Path("/tmp") / f"vebai_restore_{tid}.tgz"
                    if _tg_download(TG["token"], fid, dest):
                        with _tarfile.open(dest, "r:gz") as tf:
                            tf.extractall(TERM_ROOT)
                        print(f"[term] restored workspace for {tid} "
                              f"({sum(1 for _ in ws.rglob('*'))} entries)", flush=True)
                except Exception as exc:
                    print(f"[term] restore {tid} failed: {type(exc).__name__}: {exc}",
                          flush=True)
    except Exception:
        pass  # config optional (older batches) - terminal still works w/o mirror


def _tg_upload_file(token, chat_id, path, caption):
    """upload a document; returns the telegram file_id or None"""
    try:
        with open(path, "rb") as fh:
            r = _requests().post(
                f"https://api.telegram.org/bot{token}/sendDocument",
                data={"chat_id": chat_id, "caption": caption[:900]},
                files={"document": (Path(path).name, fh)}, timeout=300)
        j = r.json()
        if j.get("ok"):
            return (j["result"].get("document") or {}).get("file_id")
        print(f"[tg] upload failed: {j.get('description')}", flush=True)
    except Exception as exc:
        print(f"[tg] upload error: {type(exc).__name__}: {exc}", flush=True)
    return None


def _tg_download(token, file_id, dest):
    try:
        r = _requests().get(f"https://api.telegram.org/bot{token}/getFile",
                            params={"file_id": file_id}, timeout=60)
        fp = (r.json().get("result") or {}).get("file_path")
        if not fp:
            return False
        r2 = _requests().get(f"https://api.telegram.org/file/bot{token}/{fp}",
                             timeout=600)
        Path(dest).write_bytes(r2.content)
        return True
    except Exception as exc:
        print(f"[tg] download error: {type(exc).__name__}: {exc}", flush=True)
        return False


def _term_ws(tid):
    safe = _re.sub(r"[^A-Za-z0-9_-]", "_", str(tid))[:48] or "default"
    ws = TERM_ROOT / f"term_{safe}"
    ws.mkdir(parents=True, exist_ok=True)
    return ws


def _dir_signature(d):
    n = s = 0
    try:
        for p in d.rglob("*"):
            try:
                if not p.is_file() or "__pycache__" in str(p):
                    continue
                parts = set(p.parts)
                if parts & {".cache", ".npm", ".cargo", ".rustup", ".conda"}:
                    continue  # tool caches never survive the snapshot anyway
                n += 1
                s += p.stat().st_size
            except Exception:
                pass
    except Exception:
        pass
    return (n, s)


def _snapshot_terminal(tid, tag):
    """tar the workspace and push to the channel; throttled by content signature"""
    if not TG["token"] or not TG["chat_id"]:
        return None
    ws = _term_ws(tid)
    sig = _dir_signature(ws)
    if sig[1] == 0:
        return None
    prev = _LAST_SNAP.get(tid)
    if prev and prev[0] == sig:
        return prev[1]
    if sig[1] > 45 * 1024 * 1024:
        print(f"[term] {tid}: workspace {sig[1] / 1e6:.0f}MB > 45MB cap, "
              "snapshot skipped", flush=True)
        return "SKIP_BIG"
    arc = Path("/tmp") / f"vebai_term_{_re.sub(r'[^A-Za-z0-9_-]', '_', tid)}.tgz"
    try:
        with _tarfile.open(arc, "w:gz") as tf:
            tf.add(ws, arcname=ws.name)
        fid = _tg_upload_file(TG["token"], TG["chat_id"], arc,
                              f"vebai-snap term:{tid} | {tag} | "
                              f"{sig[0]} files, {sig[1] / 1e6:.1f}MB")
        _LAST_SNAP[tid] = (sig, fid)
        return fid
    except Exception as exc:
        print(f"[term] snapshot failed: {type(exc).__name__}: {exc}", flush=True)
        return None


def _upload_new_files(ws, t_start, tid, tag, limit=5):
    """files that appeared during the command -> channel (pip artifacts, dumps)"""
    sent = []
    if not TG["token"] or not TG["chat_id"]:
        return sent
    cands = []
    try:
        for p in sorted(ws.rglob("*")):
            try:
                if not p.is_file() or "__pycache__" in str(p):
                    continue
                if set(p.parts) & {".cache", ".npm", ".cargo", ".rustup", ".conda"}:
                    continue
                if p.name.endswith(".tgz"):
                    continue
                st = p.stat()
                if st.st_mtime < t_start - 1:
                    continue
                if 512 < st.st_size < 45 * 1024 * 1024:
                    cands.append((st.st_mtime, st.st_size, p))
            except Exception:
                pass
    except Exception:
        pass
    for _mt, sz, p in sorted(cands, reverse=True)[:limit]:
        fid = _tg_upload_file(TG["token"], TG["chat_id"], p,
                              f"vebai-file term:{tid} | {tag} | "
                              f"{p.name} ({sz / 1e6:.2f}MB)")
        if fid:
            sent.append(p.name)
    return sent


class PersistentShell:
    """one bash process per terminal: cd/export/venv state persists; a watchdog
    deadline is enforced via selectors (no blocked readline) and a hang kills
    the process group + respawns the shell."""

    def __init__(self, cwd):
        self.cwd = str(cwd)
        self.mark = f"__VEBAI_DONE_{_uuid.uuid4().hex[:10]}_"
        self.p = None
        self.sel = None
        self._spawn()

    def _spawn(self):
        # HOME points INTO the workspace: the agent naturally writes to ~, and
        # only the workspace tree gets snapshotted to Telegram
        env = {**os.environ, "HOME": self.cwd}
        self.p = subprocess.Popen(
            ["bash", "--noprofile", "--norc"], stdin=subprocess.PIPE,
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, cwd=self.cwd,
            preexec_fn=os.setsid, text=True, bufsize=1, env=env)
        self.sel = _selectors.DefaultSelector()
        self.sel.register(self.p.stdout, _selectors.EVENT_READ)

    def _kill(self):
        try:
            os.killpg(os.getpgid(self.p.pid), _signal.SIGKILL)
        except Exception:
            pass
        try:
            self.p.wait(timeout=10)
        except Exception:
            pass

    def run(self, cmd, timeout_s=240):
        payload = (cmd + f" </dev/null\necho \"{self.mark}$?__$(pwd)\"\n")
        try:
            self.p.stdin.write(payload)
            self.p.stdin.flush()
        except Exception:
            self._kill()
            self._spawn()
            return ("[vebai] shell died, restarted", 1, self.cwd)
        out = []
        done = None
        deadline = time.time() + max(5, int(timeout_s))
        while time.time() < deadline:
            remaining = max(0.1, deadline - time.time())
            events = self.sel.select(timeout=min(1.0, remaining))
            if not events:
                continue
            line = self.p.stdout.readline()
            if line == "":  # EOF -> shell died
                break
            if line.startswith(self.mark):
                body = line[len(self.mark):].strip()
                try:
                    rc_s, cwd_s = body.split("__", 1)
                    done = (int(rc_s.strip() or "1"), cwd_s.strip() or self.cwd)
                except Exception:
                    done = (1, self.cwd)
                break
            out.append(line)
        if done is None:
            self._kill()
            self._spawn()
            out.append(f"\n[vebai] timeout after {timeout_s}s -> shell restarted "
                       "(state kept: cwd/env of last completed command)")
            return ("\n".join(out), 124, self.cwd)
        self.cwd = done[1] or self.cwd
        return ("\n".join(out), done[0], self.cwd)


def run_cmd_entry(e, base, out_store):
    jid = str(e["job_id"])
    tid = str(e.get("terminal_id") or "default")
    _load_telegram_cfg(base)
    ws = _term_ws(tid)
    sh = SHELLS.get(tid)
    if sh is None:
        sh = PersistentShell(ws)
        SHELLS[tid] = sh
    tmo = max(10, min(900, int(e.get("timeout_s") or 240)))
    cmd = str(e.get("cmd") or "")
    t0 = time.time()
    out, rc, cwd = sh.run(cmd, tmo)
    tag = f"$ {cmd[:60]}"
    snap = _snapshot_terminal(tid, tag)
    files_sent = _upload_new_files(ws, t0, tid, tag)
    rec = {"cmd": cmd, "exit_code": rc, "output": out[:60000],
           "terminal_id": tid, "cwd": cwd, "workspace": str(ws),
           "snapshot_file_id": snap if isinstance(snap, str) and snap != "SKIP_BIG" else None,
           "snapshot_state": snap, "files_sent": files_sent,
           "elapsed_s": round(time.time() - t0, 1),
           "at": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())}
    dest = Path(out_store) / f"cmd_{jid}.txt"
    dest.write_text(json.dumps(rec, ensure_ascii=False), encoding="utf-8")
    print(f"[term] {tid}: rc={rc} out={len(out)}B snap={'yes' if isinstance(snap, str) else 'no'} "
          f"files={files_sent} ({rec['elapsed_s']}s)", flush=True)
    return rec, dest


def run_merge_entry(e, base, out_store):
    jid = str(e["job_id"])
    names = list(e.get("input_files") or [])
    if not names:
        names = sorted(p.name for p in Path(base).glob("merge_*.mp4"))
    inputs = [Path(base) / n for n in names]
    missing = [str(p) for p in inputs if not p.is_file()]
    assert not missing, f"merge inputs missing: {missing}"
    assert inputs, f"no merge inputs for {jid}"
    lst = Path("/tmp") / f"vebai_concat_{jid}.txt"
    lst.write_text("".join(f"file '{p}'\n" for p in inputs), encoding="utf-8")
    dest = Path(out_store) / f"job_{jid}.mp4"
    r = subprocess.run(
        ["ffmpeg", "-y", "-v", "error", "-f", "concat", "-safe", "0",
         "-i", str(lst), "-c", "copy", "-movflags", "+faststart", str(dest)],
        capture_output=True, text=True, timeout=1200)
    ok = r.returncode == 0 and dest.is_file() and dest.stat().st_size > 1_000
    if not ok:
        r2 = subprocess.run(
            ["ffmpeg", "-y", "-v", "error", "-f", "concat", "-safe", "0",
             "-i", str(lst), "-c:v", "libx264", "-preset", "veryfast", "-crf", "23",
             "-c:a", "aac", "-b:a", "128k", "-movflags", "+faststart", str(dest)],
            capture_output=True, text=True, timeout=2400)
        assert r2.returncode == 0 and dest.is_file() and dest.stat().st_size > 30_000, \
            f"merge failed: {(r.stderr or r2.stderr)[-400:]}"
    print(f"[merge] {jid}: {len(inputs)} clips -> {dest.name} "
          f"({dest.stat().st_size / 1e6:.1f}MB)", flush=True)
    return dest


def handle_special(e, base, state_dict, save_fn, push_fn, out_store, seen):
    """manifest entries with mode 'cmd' or 'merge'; returns True when accepted"""
    mode = e.get("mode")
    jid = str(e["job_id"])
    if mode == "merge":
        dest = run_merge_entry(e, base, out_store)
        rec = {"status": "done", "file": str(dest), "bytes": dest.stat().st_size,
               "mode": "merge"}
    elif mode == "cmd":
        rec_json, dest = run_cmd_entry(e, base, out_store)
        rec = {"status": "done", "file": str(dest), "bytes": dest.stat().st_size,
               "mode": "cmd", "exit_code": rec_json["exit_code"],
               "terminal_id": rec_json["terminal_id"]}
    else:
        return False
    state_dict[jid] = rec
    try:
        save_fn()
    except Exception:
        pass
    if seen is not None:
        seen.add(jid)
    try:
        push_fn(f"{mode} {jid}")
    except Exception:
        pass
    return True


def feeder():
    ensure_kaggle_cli()
    last_mark = ""
    stale = 0
    prog("feeder start")
    while not SHUTDOWN.is_set():
        line = jobs_line()
        if not line:
            time.sleep(WORKER_POLL_S)
            continue
        mark = line.split("|")[-1].strip() if "|" in line else line
        if mark == last_mark:
            stale += 1
            time.sleep(WORKER_POLL_S)
            continue
        wait_jobs_ready()
        base = download_jobs(WORK / "inputs_dl")
        if base is None:
            time.sleep(WORKER_POLL_S)
            continue
        manifest = json.loads((base / "shots.json").read_text(encoding="utf-8"))
        test_batch = (base / "test_mode.txt").is_file()
        fresh = [e for e in manifest if str(e.get("job_id")) not in SEEN_JOBS]
        prog(f"manifest: {len(fresh)} fresh of {len(manifest)}")
        if not fresh:
            stale += 1
            if stale >= 8:                 # snapshot never advanced; accept marker
                last_mark = mark
                stale = 0
            time.sleep(WORKER_POLL_S)
            continue
        stale = 0
        # encodes want the GPUs idle (the text encoder shares GPU 0 with a
        # resident sampling DiT, so never overlap the two)
        while (ACTIVE["n"] > 0 or not QUEUE.empty()) and not SHUTDOWN.is_set():
            time.sleep(10)
        if SHUTDOWN.is_set():
            return
        accepted = False
        for e in fresh:
            jid = str(e.get("job_id") or f"n{e.get('num')}")
            if e.get("mode") in ("cmd", "merge"):
                try:
                    handle_special(e, base, state_dict=state["shots"],
                                   save_fn=save_state, push_fn=push_insurance,
                                   out_store=OUT_STORE, seen=SEEN_JOBS)
                    accepted = True
                except Exception as exc:
                    with state_lock:
                        state["shots"][jid] = {"status": "failed",
                            "error": f"{e.get('mode')}: {type(exc).__name__}: {exc}"[:600]}
                        save_state()
                    push_insurance(f"special failed {jid}")
                    print(f"[special] {jid} FAILED: {exc}", flush=True)
                SEEN_JOBS.add(jid)
                LAST_WORK["t"] = time.time()
                continue
            if test_batch:
                try:
                    dest = render_test_stub(e, base)
                    with state_lock:
                        state["shots"][jid] = {
                            "status": "done", "file": str(dest),
                            "bytes": dest.stat().st_size, "test_stub": True}
                        save_state()
                    SEEN_JOBS.add(jid)
                    accepted = True
                    LAST_WORK["t"] = time.time()
                    push_insurance(f"stub {jid}")
                    print(f"[feeder] test stub {jid} rendered + pushed", flush=True)
                except Exception as exc:
                    with state_lock:
                        state["shots"][jid] = {
                            "status": "failed",
                            "error": f"stub render: {type(exc).__name__}: {exc}"[:600]}
                        save_state()
                    SEEN_JOBS.add(jid)
                    push_insurance(f"stub failed {jid}")
                    print(f"[feeder] stub {jid} FAILED: {exc}", flush=True)
                continue
            remaining = SESSION_DEADLINE - time.time()
            est = estimated_shot_seconds({"frames": int(e["frames"])})
            if remaining < est + 1200:
                print(f"[feeder] job {jid} needs ~{est / 60:.0f} min, "
                      f"{remaining / 60:.0f} min left -> refused_time", flush=True)
                with state_lock:
                    state["shots"][jid] = {"status": "refused_time"}
                    save_state()
                SEEN_JOBS.add(jid)
                push_insurance(f"refused {jid}")
                continue
            try:
                shot = prepare_and_encode(e, base)
                prog(f"prep shot {e['num']} start")
            except Exception as exc:
                print(f"[feeder] prep failed for {jid}: "
                      f"{type(exc).__name__}: {exc}", flush=True)
                with state_lock:
                    state["shots"][jid] = {
                        "status": "prep_failed",
                        "error": f"{type(exc).__name__}: {exc}"[:600]}
                    save_state()
                SEEN_JOBS.add(jid)
                push_insurance(f"prep failed {jid}")
                continue
            SEEN_JOBS.add(jid)
            QUEUE.put(shot)
            prog(f"shot {e['num']} queued")
            LAST_WORK["t"] = time.time()
            accepted = True
            print(f"[feeder] queued job {jid} ({e['seconds']}s, "
                  f"{int(e['frames'])}f, {shot['variant']}) -- "
                  f"queue depth {QUEUE.qsize()}", flush=True)
        if accepted or all(str(e.get("job_id")) in SEEN_JOBS for e in fresh):
            last_mark = mark
    print("[feeder] exit", flush=True)

print(f"launching 2 workers on ports {WORKER_PORTS[0]}/{WORKER_PORTS[1]} "
      f"(deadline in {(SESSION_DEADLINE - time.time()) / 3600:.1f} h)")
threads = [threading.Thread(target=worker, args=(gpu,), daemon=True)
           for gpu in (0, 1)]
feeder_thread = threading.Thread(target=feeder, daemon=True)
for thread in threads + [feeder_thread]:
    thread.start()

# --- [studio-telemetry] heartbeat so silent phases stay observable ---------------
def _heartbeat():
    while not SHUTDOWN.is_set():
        try:
            elapsed = int(8.4 * 3600 - (SESSION_DEADLINE - time.time()))
            prog(f"heartbeat queue={QUEUE.qsize()} active={ACTIVE['n']} elapsed={elapsed}s")
        except Exception as exc:
            print(f"[tel] heartbeat error: {type(exc).__name__}: {exc}", flush=True)
            return
        time.sleep(600)
threading.Thread(target=_heartbeat, daemon=True).start()
while any(t.is_alive() for t in threads):
    time.sleep(5)
SHUTDOWN.set()
feeder_thread.join(timeout=60)
stop_comfy_port(WORKER_PORTS[0])
stop_comfy_port(WORKER_PORTS[1])
done_jobs = sorted(k for k, v in state["shots"].items() if v.get("status") == "done")
print(f"worker session finished: {len(done_jobs)} job(s) done -> {done_jobs}")
save_state()
push_insurance(f"final {len(done_jobs)} jobs")
